# FOC Current-Controlled Wheel-Slip Simulator

This separate notebook models the project's four-wheel omni robot with wheel slip and a simplified FOC current loop. The robot-level PID requests per-wheel q-axis current; an inner current controller generates voltage, actual current follows the motor electrical equation, and that actual current drives wheel torque. The ideal-current model remains in the original notebook.

## Notebook map

1. **Setup and assumptions:** project physical parameters plus FOC voltage and current-loop settings.
2. **Reference trajectory:** route geometry, timing, velocity/acceleration references, and obstacles.
3. **Wheel geometry and contact:** self-contained wheel frames, slip kinematics, roller-gap geometry, and friction law.
4. **Coupled plant:** integrate chassis and wheel mechanics using actual motor current.
5. **FOC actuator:** track commanded `Iq_ref` through a current PI loop, q-axis voltage saturation, and winding R/L dynamics.
6. **Simulation and diagnostics:** compare commanded current with actual current, q-axis voltage, wheel speed, slip, and force balance.

## Model boundary

- Physical simulator state is `[x_world, y_world, phi, vx_world, vy_world, omega, theta_dot_1, ..., theta_dot_4]`. Wheel speeds are independent states; wheel angles are not included.
- The active `WheelSlipRobotSimulator` evaluates the notebook's wheel-ground d/a slip and force laws. Drive force acts on the chassis and its matching contact torque appears in that wheel's rotation equation.
- The prior effective-mass matrix remains in `RollingOnlyRobotSimulator` as a no-slip baseline only. The slip-capable plant uses the rigid-body mass matrix and separately integrates wheel inertia, avoiding a second reflection of wheel inertia into the chassis matrix.
- Wheel contact uses the smooth friction law implemented in the notebook, with configured friction parameters and normal load. It transitions smoothly from the configured static coefficient near zero slip toward the dynamic coefficient as slip grows; it is not an ideal stick/slip complementarity solver.
- `DT` is the controller sample time. The contact dynamics are stiff near zero slip, so the plant uses adaptive RK45 integration within each fixed controller interval. Its tolerances and maximum step are numerical settings that should be checked for convergence, not hardware timing claims.
- Optional wheel-load redistribution is a repeatable surrogate that conserves total robot weight, not a ZMP, suspension, or wheel-hop model. The body-frame external wrench and obstacle models remain separate.
- The outer robot PID requests q-axis current `Iq_ref`; it does not command motor voltage or wheel speed. The inner FOC current PI computes average q-axis voltage `Vq`, limited by a configurable bus/modulation approximation.
- Actual q-axis current `Iq` is an actuator state. It follows `Lq * dIq/dt = Vq - Rw * Iq - Ke * omega_m`; motor torque is `Kt * Iq`. The mechanical plant uses actual interval-average current, not `Iq_ref`.
- This surface-PM, `Id = 0` equivalent omits d-axis current, cross-coupling, PWM switching, bus sag, and detailed inverter behavior. The q-axis voltage limit and current-loop bandwidth are explicit modeling assumptions, not hardware measurements.
- Direct drive is assumed, so motor mechanical speed equals wheel angular speed. Project friction and motor values remain provisional until measured or verified.

## Reference documents

`COCO_Project_Report1_anonymized.pdf` is a drone-racing RTI/SQP NMPC example, not this robot's physics model. Its transferable ideas are stage-wise trajectory tracking, explicit constraints, warm starts, and solver-latency awareness. This notebook uses a locally linearized prediction of its nonlinear wheel-slip plant; the QP prediction is an approximation, while the plant evaluates the nonlinear contact law.


## 1. Import Required Libraries

Import the project’s robot constants and the Python libraries used for simulation and plotting. NumPy and Matplotlib are required; CVXPY is used in the later optional QP section.

In [ ]:
import os

# Cap native numerical-library threading. Run this setup cell before importing NumPy.
# The simulator's time-stepping loop is sequential; this limits BLAS/OpenMP threads
# but does not parallelize a single trajectory.
N_CORES = 2
for _thread_env in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_env] = str(N_CORES)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import importlib
import robot_constants as _robot_constants

# Refresh cached project constants when this setup cell is rerun.
_robot_constants = importlib.reload(_robot_constants)
from robot_constants import (
    ALPHA_A,
    ALPHA_D,
    F_N,
    GRAVITY,
    IQ_LIMIT,
    K_SMOOTH_A,
    K_SMOOTH_D,
    MOTOR_KT,
    MOTOR_KV,
    MOTOR_RESISTANCE,
    MOTOR_INDUCTANCE,
    MU_A_DYN,
    MU_A_STAT,
    MU_D_STAT,
    MU_D_DYN,
    ROBOT_INERTIA,
    ROBOT_MASS,
    ROLLER_PITCH,
    ROLLER_WIDTH_DTHETA,
    WHEEL_ALPHA_DEG,
    WHEEL_INERTIA,
    WHEEL_RADIUS,
    WHEEL_ROBOT_DIST,
)

# --------------------------- Simulation configuration ---------------------------
# Use PID by default while validating the nonlinear plant; set to "MPC" to run the QP.
CONTROLLER_MODE = "PID"
MPC_HORIZON = 8           # Current feasibility-tested LTV prediction horizon.
MPC_CONSTRAINT_TOL = 1e-4  # Match OSQP absolute tolerance when reporting feasibility.
# FOC electrical model assumptions. The full-charge pack is 16.8 V; for this
# amplitude-invariant q-axis convention, use Vdc/sqrt(3) as a first voltage-vector limit.
MOTOR_BUS_VOLTAGE = 16.8  # V, nominal full-charge pack value supplied by the user.
FOC_Q_AXIS_VOLTAGE_LIMIT = MOTOR_BUS_VOLTAGE / np.sqrt(3.0)  # provisional modulation mapping.
MOTOR_KE = 60.0 / (2.0 * np.pi * MOTOR_KV)  # V*s/rad from Kv in rpm/V; confirm datasheet convention.
FOC_CURRENT_LOOP_BANDWIDTH_HZ = 500.0  # provisional PI tuning parameter; replace with drive data.
FOC_ELECTRICAL_MAX_STEP = 1e-4  # s; resolve the L/R electrical transient.
DT = 0.01
MAX_ACCELERATION = 5.5  # m/s^2; below the modeled current/traction acceleration capability
MAX_VELOCITY = 4.5      # m/s; fast soccer-robot transit target, still actuator-limited
MAX_YAW_RATE = 3.0      # rad/s; provisional until measured
N_WHEELS = len(WHEEL_ALPHA_DEG)

# F_N is the nominal normal load on one wheel. MU_D_STAT * F_N is the
# corresponding per-wheel drive-traction threshold for equal load sharing.
F_ROLL_LIMIT = MU_D_STAT * F_N

# Optional surrogate for unknown wheel-to-wheel load variation. Its amplitude
# grows with robot speed and its fixed seed makes each run reproducible.
ENABLE_WHEEL_LOAD_DISTURBANCE = True
WHEEL_LOAD_DISTURBANCE_STRENGTH = 0.55  # maximum fraction away from equal load
WHEEL_LOAD_REFERENCE_SPEED = 2.5        # disturbance reaches full strength here [m/s]
WHEEL_LOAD_CORRELATION_TIME = 0.15      # keep load changes correlated over time [s]
WHEEL_LOAD_RANDOM_SEED = 7


def make_speed_scaled_wheel_load_disturbance(
    n_wheels=N_WHEELS, total_weight=ROBOT_MASS * GRAVITY,
    max_fraction=WHEEL_LOAD_DISTURBANCE_STRENGTH,
    reference_speed=WHEEL_LOAD_REFERENCE_SPEED,
    correlation_time=WHEEL_LOAD_CORRELATION_TIME,
    seed=WHEEL_LOAD_RANDOM_SEED,
):
    """Create a repeatable disturbance returning nonnegative wheel loads [N].

    The four loads always sum to robot weight. A centered, slowly varying
    random pattern redistributes load between wheels; its amplitude increases
    linearly with speed until ``reference_speed``. This is a tunable surrogate
    for unmeasured wheel bounce, not a ZMP or suspension model.
    """
    if n_wheels < 1 or total_weight <= 0.0:
        raise ValueError("n_wheels and total_weight must be positive")
    if not 0.0 <= max_fraction < 1.0:
        raise ValueError("max_fraction must be in [0, 1) to keep every wheel loaded")
    if reference_speed <= 0.0:
        raise ValueError("reference_speed must be positive")
    rng = np.random.default_rng(seed)
    random_pattern = np.zeros(n_wheels, dtype=float)
    nominal_load = total_weight / n_wheels

    def sample_normal_loads(speed_m_s, dt_s):
        nonlocal random_pattern
        if correlation_time <= 0.0:
            persistence = 0.0
        else:
            persistence = float(np.exp(-max(dt_s, 0.0) / correlation_time))
        random_pattern = (
            persistence * random_pattern
            + np.sqrt(max(0.0, 1.0 - persistence**2)) * rng.normal(size=n_wheels)
        )
        random_pattern -= np.mean(random_pattern)  # conserve total load
        peak = np.max(np.abs(random_pattern))
        if peak > 1e-12:
            random_pattern /= peak
        speed_scale = np.clip(max(float(speed_m_s), 0.0) / reference_speed, 0.0, 1.0)
        load_fraction = max_fraction * speed_scale * random_pattern
        # The configured max_fraction is below one, so no wheel can go negative.
        return nominal_load * (1.0 + load_fraction)

    return sample_normal_loads

# The project friction law is bounded by its configured static coefficient.
# Retain the bound as a safety check on the computed contact force.
ALLOW_DRIVE_TRACTION_OVERLOAD = False

# The configured friction smoothing gains make contact dynamics stiff near zero slip.
# The plant uses implicit backward-Euler microsteps; halve this step for convergence checks.
CONTACT_INTEGRATION_MAX_STEP = 1e-3  # s
CURRENT_FEASIBILITY_BISECTION_STEPS = 14  # conservative scale resolution: 1 / 2^14
SLIP_THRESHOLD = 0.019261  # m/s, diagnostic threshold used to flag noticeable drive slip


## 2. Reference Targets: Position, Velocity, and Heading at Each Time

This notebook does not ask the MPC to invent a route. `ROUTE_WAYPOINTS` is a manually selected centerline that already passes on chosen sides of the static obstacles. The code parameterizes that route by time and supplies a desired position and velocity at every MPC stage. These references are fixed data; the QP chooses currents and predicted robot states to track them.

The route's forward/backward speed envelope accelerates from rest, stays under configured speed and acceleration limits, and brakes to rest at the last waypoint. That makes the endpoint a receiving-pass style target: goal position with zero translational velocity. The path itself is not the convexity claim.

Convexity comes from the optimization problem: at each horizon stage the obstacle side is fixed from the nominal reference, so obstacle avoidance is a linear half-space. State updates are affine and the tracking/effort costs are quadratic. The QP cannot decide to pass either side of an obstacle in the same solve; changing sides is an outer route-choice decision. The drone report uses a fixed spline and nonlinear SQP/RTI tracking, and says trajectory planning is outside its scope.

Heading is another time-indexed target. Use the route tangent during transit, a fixed target point for pass/shot behavior, or provide a custom callback for a scheduled heading. For a receiving pass, set the endpoint velocity to zero and choose the heading toward the passer or intended shot target.


**Drive-slip diagnostic:** The active slip plant stores actual wheel speeds and simulates d-axis wheelspin with the project friction law. `ALLOW_DRIVE_TRACTION_OVERLOAD` bypasses the controller feasibility check against the configured static-friction capacity; the contact force remains governed by the notebook's static-to-dynamic friction law.


In [ ]:
from scipy.interpolate import CubicSpline

# Define the smooth, timed robot route first. Static obstacles are placed beside
# this path below, with enough clearance for the PID plant trajectory.
FIELD_LENGTH = 4.5  # x direction [m]
FIELD_WIDTH = 6.0   # y direction [m]
# Two joined elliptical loops form a broad figure-eight that fits inside the field.
# The lower loop runs counterclockwise; the upper loop runs clockwise. They meet
# at the center with a continuous tangent, so the robot swerves through the crossing.
FIGURE_EIGHT_RADIUS_X = 1.50
FIGURE_EIGHT_RADIUS_Y = 1.25
LOOP_PHASE = np.linspace(0.0, 2.0 * np.pi, 121)
LOWER_PHASE = 0.5 * np.pi + LOOP_PHASE
UPPER_PHASE = 1.5 * np.pi - LOOP_PHASE
LOWER_LOOP = np.column_stack((
    FIGURE_EIGHT_RADIUS_X * np.cos(LOWER_PHASE),
    -FIGURE_EIGHT_RADIUS_Y + FIGURE_EIGHT_RADIUS_Y * np.sin(LOWER_PHASE),
))
UPPER_LOOP = np.column_stack((
    FIGURE_EIGHT_RADIUS_X * np.cos(UPPER_PHASE),
    FIGURE_EIGHT_RADIUS_Y + FIGURE_EIGHT_RADIUS_Y * np.sin(UPPER_PHASE),
))
ROUTE_WAYPOINTS = np.vstack((LOWER_LOOP[:-1], UPPER_LOOP[:-1], LOWER_LOOP[0]))
ROUTE_WAYPOINTS[-1] = ROUTE_WAYPOINTS[0]  # exact endpoint match for the periodic spline

# The path is defined first. These small obstacles sit in opposite field corners,
# beside the figure-eight rather than inside the path. Clearance is checked each step.
STATIC_OBSTACLES = [
    np.array([[1.85, 2.55], [2.15, 2.55], [2.15, 2.85], [1.85, 2.85]]),
    np.array([[-2.15, -2.85], [-1.85, -2.85], [-1.85, -2.55], [-2.15, -2.55]]),
]
# Each dynamic obstacle stores polygon vertices in its local frame and a function
# returning its predicted world-frame translation at time_s. Example:
# {"vertices": local_rectangle, "position_fn": lambda time_s: np.array([x0+vx*time_s, y0+vy*time_s])}
DYNAMIC_OBSTACLES = []
ROBOT_CLEARANCE = 0.35  # m; footprint + safety margin + prediction/discretization margin
# Shape timed targets to the current-limited PID plant so they remain trackable.
# Combined reference acceleration is sqrt(2.0^2 + 1.2^2) ~= 2.33 m/s^2.
REFERENCE_MAX_SPEED = 2.2  # m/s; the plant reaches about 1.7 m/s on this loop
REFERENCE_LATERAL_ACCELERATION = 2.0  # m/s^2; below the observed 2.5 m/s^2 slip onset
REFERENCE_TANGENTIAL_ACCELERATION = 1.2  # m/s^2; creates visible speed-up/braking phases


def convex_polygon_halfspaces(vertices):
    """Return unit outward edge normals n and bounds b for a CCW convex polygon."""
    vertices = np.asarray(vertices, dtype=float)
    edges = np.roll(vertices, -1, axis=0) - vertices
    normals = np.column_stack((edges[:, 1], -edges[:, 0]))
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)
    return normals, np.sum(normals * vertices, axis=1)


def obstacle_safe_halfspace(route_point, vertices, clearance=ROBOT_CLEARANCE):
    """Choose the locally safe side of a convex obstacle as n dot p >= b+clearance.

    The side is selected from the route point and held fixed in the local QP.
    Obstacle-side selection itself is discrete/nonconvex; the resulting halfspace
    is affine and convex for that selected route corridor.
    """
    route_point = np.asarray(route_point, dtype=float)
    vertices = np.asarray(vertices, dtype=float)
    normals, _ = convex_polygon_halfspaces(vertices)
    closest_point = None
    edge_normal = None
    min_distance = np.inf
    for edge_index, start in enumerate(vertices):
        end = vertices[(edge_index + 1) % len(vertices)]
        edge = end - start
        fraction = np.clip(np.dot(route_point - start, edge) / np.dot(edge, edge), 0.0, 1.0)
        candidate = start + fraction * edge
        distance = np.linalg.norm(route_point - candidate)
        if distance < min_distance:
            min_distance = distance
            closest_point = candidate
            edge_normal = normals[edge_index]
    normal = edge_normal if min_distance < 1e-10 else (route_point - closest_point) / min_distance
    return normal, float(normal @ closest_point + clearance)


def point_polygon_distance(point, vertices):
    """Return Euclidean clearance from a point to a convex polygon boundary/interior."""
    point = np.asarray(point, dtype=float)
    vertices = np.asarray(vertices, dtype=float)
    normals, bounds = convex_polygon_halfspaces(vertices)
    if np.all(normals @ point <= bounds):
        return 0.0
    distances = []
    for edge_index, start in enumerate(vertices):
        edge = vertices[(edge_index + 1) % len(vertices)] - start
        fraction = np.clip(np.dot(point - start, edge) / np.dot(edge, edge), 0.0, 1.0)
        distances.append(np.linalg.norm(point - (start + fraction * edge)))
    return float(min(distances))


def obstacle_polygons_at(time_s):
    """Return named world-frame obstacle polygons and required clearances at time_s."""
    obstacles = [
        (f"static_{index}", np.asarray(vertices, dtype=float), ROBOT_CLEARANCE)
        for index, vertices in enumerate(STATIC_OBSTACLES)
    ]
    for index, obstacle in enumerate(DYNAMIC_OBSTACLES):
        translation = np.asarray(obstacle["position_fn"](time_s), dtype=float)
        if translation.shape != (2,):
            raise ValueError("dynamic obstacle position_fn must return [x, y]")
        vertices = np.asarray(obstacle["vertices"], dtype=float) + translation
        obstacles.append((
            obstacle.get("name", f"dynamic_{index}"), vertices,
            float(obstacle.get("clearance", ROBOT_CLEARANCE)),
        ))
    return obstacles


def obstacle_clearance_report(position, time_s):
    """Measure center-to-polygon clearance after a plant step; negative margin is collision."""
    report = []
    for name, vertices, required_clearance in obstacle_polygons_at(time_s):
        distance = point_polygon_distance(position, vertices)
        report.append({
            "name": name,
            "distance_m": distance,
            "required_clearance_m": required_clearance,
            "margin_m": distance - required_clearance,
        })
    return report


def pid_obstacle_repulsion(position, time_s, gain=4.0, influence_distance=1.0):
    """Add a smooth, local repulsive acceleration for the PID comparison baseline."""
    acceleration = np.zeros(2)
    for _, vertices, required_clearance in obstacle_polygons_at(time_s):
        distance = point_polygon_distance(position, vertices)
        gap = distance - required_clearance
        if gap < influence_distance:
            normal, _ = obstacle_safe_halfspace(position, vertices, clearance=0.0)
            acceleration += gain * (influence_distance - max(gap, 0.0)) * normal
    return acceleration


# Fit the centerline by chord length, then reparameterize samples by measured arc length.
_route_chords = np.linalg.norm(np.diff(ROUTE_WAYPOINTS, axis=0), axis=1)
_route_knots = np.r_[0.0, np.cumsum(_route_chords)]
_route_x = CubicSpline(_route_knots, ROUTE_WAYPOINTS[:, 0], bc_type="periodic")
_route_y = CubicSpline(_route_knots, ROUTE_WAYPOINTS[:, 1], bc_type="periodic")
_route_parameter = np.linspace(0.0, _route_knots[-1], 5001)
_route_points = np.column_stack((_route_x(_route_parameter), _route_y(_route_parameter)))
PATH_ARC = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(_route_points, axis=0), axis=1))]
_route_dx = _route_x(_route_parameter, 1)
_route_dy = _route_y(_route_parameter, 1)
_route_ddx = _route_x(_route_parameter, 2)
_route_ddy = _route_y(_route_parameter, 2)
_route_tangent_norm = np.maximum(np.hypot(_route_dx, _route_dy), 1e-9)
PATH_TANGENT = np.column_stack((_route_dx / _route_tangent_norm, _route_dy / _route_tangent_norm))
PATH_CURVATURE = (_route_dx * _route_ddy - _route_dy * _route_ddx) / np.maximum(_route_tangent_norm**3, 1e-9)

# Build a timed centerline speed profile. Curvature and yaw rate limit corner
# speed; forward/backward passes leave room to accelerate and brake at the endpoints.
LATERAL_ACCEL_LIMIT = REFERENCE_LATERAL_ACCELERATION
TANGENTIAL_ACCEL_LIMIT = REFERENCE_TANGENTIAL_ACCELERATION
PATH_SPEED_CAP = np.minimum.reduce((
    np.full_like(PATH_CURVATURE, REFERENCE_MAX_SPEED),
    np.sqrt(LATERAL_ACCEL_LIMIT / np.maximum(np.abs(PATH_CURVATURE), 1e-6)),
    MAX_YAW_RATE / np.maximum(np.abs(PATH_CURVATURE), 1e-6),
))
PATH_SPEED = PATH_SPEED_CAP.copy()
PATH_SPEED[0] = 0.0
PATH_SPEED[-1] = 0.0
PATH_DS = np.diff(PATH_ARC)
for _ in range(2):
    for index in range(1, len(PATH_SPEED)):
        PATH_SPEED[index] = min(
            PATH_SPEED[index],
            np.sqrt(max(0.0, PATH_SPEED[index - 1]**2 + 2.0 * TANGENTIAL_ACCEL_LIMIT * PATH_DS[index - 1])),
        )
    for index in range(len(PATH_SPEED) - 2, -1, -1):
        PATH_SPEED[index] = min(
            PATH_SPEED[index],
            np.sqrt(max(0.0, PATH_SPEED[index + 1]**2 + 2.0 * TANGENTIAL_ACCEL_LIMIT * PATH_DS[index])),
        )
# Approximate each interval travel time from its arc length and endpoint speeds.
# PATH_TIME maps route distance onto clock time for the controller references.
PATH_DT = 2.0 * PATH_DS / np.maximum(PATH_SPEED[:-1] + PATH_SPEED[1:], 1e-6)
PATH_TIME = np.r_[0.0, np.cumsum(PATH_DT)]
PATH_END_TIME = float(PATH_TIME[-1])
PATH_TANGENTIAL_ACCEL = np.gradient(PATH_SPEED**2 / 2.0, PATH_ARC)
PATH_NORMAL = np.column_stack((-PATH_TANGENT[:, 1], PATH_TANGENT[:, 0]))


def trajectory_reference(time_s):
    """Return desired world position, velocity, and acceleration at time_s.

    These are fixed references, not optimization variables. The MPC tracks
    them; after PATH_END_TIME the desired robot remains at the goal and stopped.
    """
    if time_s <= 0.0:
        arc_position, speed, tangential_acceleration = PATH_ARC[0], PATH_SPEED[0], PATH_TANGENTIAL_ACCEL[0]
    elif time_s >= PATH_END_TIME:
        # Receiving-pass target: stay at the goal with zero planar velocity.
        return ROUTE_WAYPOINTS[-1].copy(), np.zeros(2), np.zeros(2)
    else:
        arc_position = np.interp(time_s, PATH_TIME, PATH_ARC)
        speed = np.interp(arc_position, PATH_ARC, PATH_SPEED)
        tangential_acceleration = np.interp(arc_position, PATH_ARC, PATH_TANGENTIAL_ACCEL)

    position = np.array([
        np.interp(arc_position, PATH_ARC, _route_points[:, 0]),
        np.interp(arc_position, PATH_ARC, _route_points[:, 1]),
    ])
    tangent = np.array([
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 0]),
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 1]),
    ])
    tangent /= max(np.linalg.norm(tangent), 1e-9)
    normal = np.array([-tangent[1], tangent[0]])
    curvature = np.interp(arc_position, PATH_ARC, PATH_CURVATURE)
    velocity = speed * tangent
    acceleration = tangential_acceleration * tangent + speed**2 * curvature * normal
    return position, velocity, acceleration


def body_to_world(vector_body, phi):
    """Rotate a body-frame vector into the world frame."""
    cosine, sine = np.cos(phi), np.sin(phi)
    return np.array([[cosine, -sine], [sine, cosine]]) @ np.asarray(vector_body, dtype=float)


def world_to_body(vector_world, phi):
    """Rotate a world-frame vector into the body frame, matching simulation.py."""
    cosine, sine = np.cos(phi), np.sin(phi)
    return np.array([[cosine, sine], [-sine, cosine]]) @ np.asarray(vector_world, dtype=float)


def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def tangent_heading_reference(time_s, position, velocity):
    """Face along the route tangent, including when the target speed is zero.

    Heading cannot be recovered from velocity at rest: the velocity direction is
    undefined there. Instead, read the geometric route tangent at this reference time.
    """
    # Clamp before/after the route so the robot holds the first/last route heading.
    clamped_time = np.clip(time_s, 0.0, PATH_END_TIME)
    arc_position = np.interp(clamped_time, PATH_TIME, PATH_ARC)
    tangent = np.array([
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 0]),
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 1]),
    ])
    return float(np.arctan2(tangent[1], tangent[0]))


def target_heading_reference(target_world):
    """Create a heading callback that faces a fixed pass/shot target.

    At every time, the desired angle points from the robot position to target_world.
    """
    target_world = np.asarray(target_world, dtype=float)

    def heading(time_s, position, velocity):
        direction = target_world - position
        return float(np.arctan2(direction[1], direction[0]))

    return heading


HEADING_MODE = "path"  # Set to "target" to face a pass/shot target.
HEADING_TARGET_WORLD = np.array([5.0, 0.0])


def configured_heading_reference():
    """Build the selected heading reference used by PID and MPC."""
    if HEADING_MODE.lower() == "path":
        return tangent_heading_reference
    if HEADING_MODE.lower() == "target":
        return target_heading_reference(HEADING_TARGET_WORLD)
    raise ValueError("HEADING_MODE must be 'path' or 'target'")


def zero_disturbance(time_s, physical_state=None):
    """Return no external body-frame wrench [Fx, Fy, torque]."""
    return np.zeros(3)


def sinusoidal_force_disturbance(amplitude=(0.0, 0.0), frequency=1.0, torque_amplitude=0.0):
    """Create a time-varying body-frame wrench callback for disturbance tests."""
    amplitude = np.asarray(amplitude, dtype=float)
    if amplitude.shape != (2,):
        raise ValueError("amplitude must be [Fx, Fy]")

    def disturbance(time_s, physical_state=None):
        phase = 2.0 * np.pi * frequency * time_s
        return np.r_[amplitude * np.sin(phase), torque_amplitude * np.sin(phase)]

    return disturbance

## 3. Wheel geometry, slip kinematics, and contact forces

For each wheel, `drive[:, i]` is the powered drive direction, `axle[:, i]` is the passive-roller axis, and \(\mathbf r_i\) is the vector from the robot centre to the wheel centre. These directions are defined in the robot body frame.

The wheel-centre velocity is rigid-body velocity at an offset point:

\[
\mathbf v_i = \mathbf v_b + \boldsymbol\omega \times \mathbf r_i, \qquad
v_{a,i}=\mathbf a_i^T\mathbf v_i.
\]

`wheel_contact_outputs()` calculates slip and friction directly from the notebook's wheel geometry, then applies the roller/gap gate once using each wheel's current angle and sampled normal load. The wheel speeds are actual states, so d-axis slip is not constrained to zero. The friction law uses each sampled normal load directly and applies a binary contact switch based on each integrated wheel angle. Both final force components are zero in the geometric gap, so chassis forces and wheel ground-load torque use the same gated force. The contact sector uses the measured roller width and gap dimensions; rubber deflection is omitted.


In [ ]:
# Wheel geometry and contact mechanics
# ------------------------------------
# Wheel angles are measured in the body-frame x-y plane. Each wheel's axle
# direction points radially from the robot centre; its drive direction is
# perpendicular to the axle. All slip and contact-force calculations live here.

def wheel_frame(alpha_deg):
    """Return (drive_direction, axle_direction) unit vectors in the body frame."""
    alpha = np.radians(float(alpha_deg) % 360.0)
    axle = np.array([np.cos(alpha), np.sin(alpha)])
    drive = np.array([np.cos(alpha + 0.5 * np.pi), np.sin(alpha + 0.5 * np.pi)])
    return drive, axle


def wheel_directions_body():
    """Return 2xN arrays of drive and axle unit vectors in wheel order."""
    frames = [wheel_frame(alpha) for alpha in WHEEL_ALPHA_DEG]
    return np.column_stack([frame[0] for frame in frames]), np.column_stack([frame[1] for frame in frames])


def wheel_center_velocity(velocity_body, omega):
    """Return each wheel-centre velocity in the robot body frame [m/s]."""
    velocity_body = np.asarray(velocity_body, dtype=float)
    if velocity_body.shape != (2,):
        raise ValueError("velocity_body must have shape (2,)")
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    rotational_velocity = np.vstack((-omega * radius[1], omega * radius[0]))
    return velocity_body[:, None] + rotational_velocity


def wheel_has_roller_contact(theta):
    """True when wheel angle [rad] lies inside a roller-contact sector."""
    phase = (float(theta) + 0.5 * ROLLER_PITCH) % ROLLER_PITCH - 0.5 * ROLLER_PITCH
    half_width = 0.5 * np.radians(ROLLER_WIDTH_DTHETA)
    return abs(phase) <= half_width


def wheel_contact_mask(wheel_angles):
    """Return the roller-contact state of each wheel angle [rad]."""
    angles = np.asarray(wheel_angles, dtype=float)
    if angles.shape != (N_WHEELS,) or not np.all(np.isfinite(angles)):
        raise ValueError("wheel_angles must contain one finite angle per wheel")
    return np.array([wheel_has_roller_contact(angle) for angle in angles], dtype=bool)


def wheel_slip_velocities(velocity_body, omega, theta_dot_actual):
    """Return drive and passive-roller axial slip velocities [m/s]."""
    theta_dot_actual = np.asarray(theta_dot_actual, dtype=float)
    if theta_dot_actual.shape != (N_WHEELS,) or not np.all(np.isfinite(theta_dot_actual)):
        raise ValueError("theta_dot_actual must contain one finite speed per wheel")
    drive, axle = wheel_directions_body()
    center_velocity = wheel_center_velocity(velocity_body, omega)
    slip_d = np.sum(drive * center_velocity, axis=0) + WHEEL_RADIUS * theta_dot_actual
    slip_a = np.sum(axle * center_velocity, axis=0)
    return slip_d, slip_a


def wheel_contact_outputs(velocity_body, omega, theta_dot_actual, wheel_angles, normal_loads=None):
    """Return contact force vectors, d/a force components, and d/a slip velocities."""
    angles = np.asarray(wheel_angles, dtype=float)
    if angles.shape != (N_WHEELS,) or not np.all(np.isfinite(angles)):
        raise ValueError("wheel_angles must contain one finite angle per wheel")
    loads = np.full(N_WHEELS, F_N, dtype=float) if normal_loads is None else np.asarray(normal_loads, dtype=float)
    if loads.shape != (N_WHEELS,) or not np.all(np.isfinite(loads)) or np.any(loads < 0.0):
        raise ValueError("normal_loads must contain one finite nonnegative load per wheel")

    drive, axle = wheel_directions_body()
    slip_d, slip_a = wheel_slip_velocities(velocity_body, omega, theta_dot_actual)

    mu_d = MU_D_DYN + (MU_D_STAT - MU_D_DYN) * np.exp(-ALPHA_D * np.abs(slip_d))
    mu_a = MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-ALPHA_A * np.abs(slip_a))
    force_d = -loads * mu_d * (2.0 / np.pi) * np.arctan(K_SMOOTH_D * slip_d)
    force_a = -loads * mu_a * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * slip_a)

    contact = wheel_contact_mask(angles)
    force_d *= contact
    force_a *= contact
    force_vectors = (drive * force_d + axle * force_a).T
    return force_vectors, force_d, force_a, slip_d, slip_a


def drive_slip_velocity(velocity_body, omega, theta_dot_actual):
    """Calculate d-axis contact slip from chassis motion and actual wheel speeds [m/s]."""
    return wheel_slip_velocities(velocity_body, omega, theta_dot_actual)[0]


def axial_slip_velocity_body(velocity_body, omega):
    """Calculate passive-roller a-axis slip from wheel-centre velocity [m/s]."""
    return wheel_slip_velocities(velocity_body, omega, np.zeros(N_WHEELS))[1]


def axial_friction_wrench_body(velocity_body, omega, wheel_angles, normal_loads=None):
    """Sum contact-gated passive-roller forces and moments into a body wrench."""
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    _, _, force_a, _, slip_a = wheel_contact_outputs(
        velocity_body, omega, np.zeros(N_WHEELS), wheel_angles, normal_loads=normal_loads
    )
    force_vectors = axle * force_a
    total_force = np.sum(force_vectors, axis=1)
    total_torque = np.sum(radius[0] * force_vectors[1] - radius[1] * force_vectors[0])
    return np.r_[total_force, total_torque], slip_a, force_a


# Geometry and contact checks: orthogonal frames, pure rolling, and roller gaps.
_test_drive, _test_axle = wheel_directions_body()
assert np.allclose(np.linalg.norm(_test_drive, axis=0), 1.0)
assert np.allclose(np.linalg.norm(_test_axle, axis=0), 1.0)
assert np.allclose(np.sum(_test_drive * _test_axle, axis=0), 0.0, atol=1e-12)

_test_velocity_body = np.array([0.5, 0.2])
_test_omega = 0.3
_test_center_velocity = wheel_center_velocity(_test_velocity_body, _test_omega)
_test_theta_dot = -np.sum(_test_drive * _test_center_velocity, axis=0) / WHEEL_RADIUS
assert np.allclose(drive_slip_velocity(_test_velocity_body, _test_omega, _test_theta_dot), 0.0, atol=1e-10)
assert wheel_has_roller_contact(0.0)
assert not wheel_has_roller_contact(0.5 * ROLLER_PITCH)
_gap_test = wheel_contact_outputs(
    np.array([1.0, 0.0]), 0.0, np.zeros(N_WHEELS),
    np.full(N_WHEELS, 0.5 * ROLLER_PITCH),
)
assert all(np.allclose(component, 0.0) for component in _gap_test[:3])
del _test_drive, _test_axle, _test_velocity_body, _test_omega, _test_center_velocity, _test_theta_dot, _gap_test


## 4. Coupled wheel-slip plant

The mechanical plant stores world pose/velocity, four independent wheel angular speeds, and four integrated wheel angles. The FOC wrapper stores actual q-axis current separately. It evaluates the project's d/a slip and force laws, applies the binary roller/gap switch to the final force outputs, sums contact wrench, and integrates chassis motion, wheel speeds, and wheel angles using actual average motor current. Wheel inertia appears in each wheel's rotational equation.

The nonlinear one-step map uses implicit backward-Euler microsteps because the configured contact law is stiff near zero slip. The MPC finite-differences this same map and uses a local affine prediction of the nonlinear slip plant; that prediction is an approximation of the contact law.


### Wheel/contact and body equations

For wheel $i$, the project computes contact-point velocity from chassis twist and actual wheel speed. In Henry's sign convention:

$$
v_{d,i}=\mathbf{d}_i^T(\mathbf{v}_b+\boldsymbol{\omega}\times\mathbf{r}_i)+R_w\dot{\theta}_i,
\qquad
v_{a,i}=\mathbf{a}_i^T(\mathbf{v}_b+\boldsymbol{\omega}\times\mathbf{r}_i).
$$

The notebook's smooth friction law maps these slips to $F_{d,i}$ and $F_{a,i}$. Let $H_d$ and $H_a$ map drive and axle forces into body wrench. The chassis equation is

$$
M_b\dot{\mathbf{q}}+\mathbf{C}(\mathbf{q})=H_d\mathbf{F}_d+H_a\mathbf{F}_a+\mathbf{w}_{ext},
\qquad
M_b=\operatorname{diag}(m,m,I_{zz}),
\quad
\mathbf{C}=[-m\omega v,\;m\omega u,\;0]^T.
$$

The wheel torque equation follows the project's sign convention and `required_static_friction()`. The forces below are the final binary-gated outputs, so drive force and ground-load torque are both zero during a roller gap:

$$
J_w\ddot{\theta}_i=K_t I_{q,i}+R_w F_{d,i}.
$$

Thus the same gated $F_{d,i}$ drives chassis translation/yaw and contributes the signed contact torque on that wheel. Wheel inertia and wheel angle are integrated independently; the no-slip effective mass matrix is retained only for the rolling-only baseline. The controller requests $I_{q,i}^{\star}$, but the motor torque equation uses actual $I_{q,i}$ after the FOC current loop and R/L response. Implicit microsteps handle the steep near-zero-slip response.


## FOC current loop and motor electrical dynamics

The robot controller requests a current reference $I_{q,i}^{\star}$. A simplified inner current PI loop creates an average q-axis voltage, with back-EMF feedforward:

$$
V_{q,i}^{\mathrm{raw}} = R_w I_{q,i}^{\star} + K_e\dot\theta_i + K_p(I_{q,i}^{\star}-I_{q,i}) + K_i\int (I_{q,i}^{\star}-I_{q,i})\,dt,
\qquad |V_{q,i}|\le V_{q,\max}.
$$

The actual current follows

$$
L_w\dot I_{q,i}=V_{q,i}-R_w I_{q,i}-K_e\dot\theta_i,
\qquad \tau_{m,i}=K_t I_{q,i}.
$$

The wheel and chassis equations use the **actual interval-average** $I_q$, not the commanded reference. This is a one-axis surface-PM FOC approximation with $I_d=0$; it omits d/q cross-coupling and PWM switching. The current-loop bandwidth is provisional. The q-axis voltage limit uses $V_{dc}/\sqrt{3}$ as an initial space-vector-modulation approximation; the exact usable value depends on drive voltage convention and firmware. The electrical substep is smaller than the configured $L_w/R_w$ time constant.



In [ ]:
class RollingOnlyRobotSimulator:
    """No-slip baseline plant; wheel inertia is reflected through the rolling constraint.

    State order is [x_world, y_world, phi, vx_world, vy_world, omega,
    theta_dot_1, theta_dot_2, theta_dot_3, theta_dot_4]. This baseline constrains
    wheel acceleration to no-slip rolling and is not the active slip plant.
    """

    enforces_no_slip = True

    def __init__(self, dt=DT, initial_state=None, disturbance_fn=zero_disturbance,
                 disturbance_forecast=None, heading_reference_fn=None,
                 wheel_load_fn=None):
        self.dt = float(dt)
        self.mass = ROBOT_MASS
        self.izz = ROBOT_INERTIA
        self.wheel_radius = WHEEL_RADIUS
        self.wheel_inertia = WHEEL_INERTIA
        self.current_limit = IQ_LIMIT
        self.traction_limit = MU_D_STAT * F_N  # nominal, equal-load value [N/wheel]
        self.wheel_load_fn = wheel_load_fn
        if self.wheel_load_fn is None and ENABLE_WHEEL_LOAD_DISTURBANCE:
            self.wheel_load_fn = make_speed_scaled_wheel_load_disturbance()
        self.normal_loads = np.full(N_WHEELS, F_N, dtype=float)
        self.wheel_traction_limits = MU_D_STAT * self.normal_loads
        self.max_acceleration = MAX_ACCELERATION
        self.max_velocity = MAX_VELOCITY
        self.max_yaw_rate = MAX_YAW_RATE
        self.disturbance_fn = disturbance_fn
        self.disturbance_forecast = disturbance_forecast
        self.heading_reference_fn = (
            configured_heading_reference() if heading_reference_fn is None else heading_reference_fn
        )

        # Geometry is fixed in body coordinates. Each column describes one wheel.
        alpha = np.radians(np.asarray(WHEEL_ALPHA_DEG, dtype=float))
        self.drive = np.vstack((-np.sin(alpha), np.cos(alpha)))  # Drive unit vectors, shape (2, 4).
        wheel_radius_vectors = WHEEL_ROBOT_DIST * np.vstack((np.cos(alpha), np.sin(alpha)))  # r_i from body centre.
        # Signed yaw lever arm is the z component of r_i cross drive_i.
        yaw_levers = wheel_radius_vectors[0] * self.drive[1] - wheel_radius_vectors[1] * self.drive[0]
        # H maps four drive forces to [Fx, Fy, tau_z]; its transpose maps body
        # twist/acceleration to the matching wheel-direction kinematic quantities.
        self.H = np.vstack((self.drive, yaw_levers))  # (3, 4)
        # H maps body twist [u, v, omega] to wheel drive speeds and yaw lever arms.
        # It is the matrix form of the old wheel-geometry force/moment sums.
        # Rigid-body mass matrix for body twist q = [u, v, omega]. Translation
        # uses mass [kg]; yaw uses Izz [kg m^2].
        self.body_mass = np.diag([self.mass, self.mass, self.izz])

        # Under no slip, theta_ddot = -(H.T @ q_dot) / R_w. Substituting this
        # into wheel torque balance reflects wheel inertia into body motion:
        # M_eff = M_body + (J_w / R_w^2) H H.T. This is the model's key constraint.
        self.effective_mass = self.body_mass + (
            self.wheel_inertia / self.wheel_radius**2
        ) * (self.H @ self.H.T)
        self.mass_inverse = np.linalg.inv(self.effective_mass)  # M_eff^-1 for the force balance.

        # Motor torque is tau_m = K_t * Iq. G maps four wheel currents to the
        # three body-twist derivatives after applying the no-slip wheel constraint.
        self.G = -(MOTOR_KT / self.wheel_radius) * self.mass_inverse @ self.H

        facet_angles = 2.0 * np.pi * np.arange(8) / 8.0
        self.facets = np.column_stack((np.cos(facet_angles), np.sin(facet_angles)))
        self.acceleration_limit_poly = self.max_acceleration * np.cos(np.pi / 8.0)
        self.velocity_limit_poly = self.max_velocity * np.cos(np.pi / 8.0)

        if initial_state is None:
            # Avoid subclass reference_state overrides during base initialization.
            self.state = RollingOnlyRobotSimulator.reference_state(self, 0.0)
        else:
            self.state = np.asarray(initial_state, dtype=float).copy()
            if self.state.shape == (6,):
                velocity_body = world_to_body(self.state[3:5], self.state[2])
                twist_body = np.r_[velocity_body, self.state[5]]
                self.state = np.r_[self.state, -(self.H.T @ twist_body) / self.wheel_radius]
            if self.state.shape != (10,):
                raise ValueError("initial_state must contain 6 robot states, optionally followed by 4 wheel angular velocities")
        self.initial_state = self.state.copy()
        self.index = 0
        self.history = []

    def heading_ref(self, time_s):
        position, velocity, _ = trajectory_reference(time_s)
        return float(self.heading_reference_fn(time_s, position, velocity))

    def yaw_rate_ref(self, time_s):
        eps = max(1e-4, self.dt * 0.25)
        return wrap_angle(self.heading_ref(time_s + eps) - self.heading_ref(time_s - eps)) / (2.0 * eps)

    def reference_state(self, time_s):
        position, velocity, _ = trajectory_reference(time_s)
        phi = self.heading_ref(time_s)
        omega = self.yaw_rate_ref(time_s)
        velocity_body = world_to_body(velocity, phi)
        theta_dot = -(self.H.T @ np.r_[velocity_body, omega]) / self.wheel_radius
        return np.r_[position, phi, velocity, omega, theta_dot]

    @staticmethod
    def coriolis(twist_body, mass):
        u, v, omega = twist_body
        return np.array([-mass * omega * v, mass * omega * u, 0.0])

    def physical_dynamics(self, state_world, current, wrench_body):
        """Apply motor/contact/disturbance forces and return world-state rates.

        Velocity is rotated to body axes for the wheel model, the body force
        balance is solved there, and translational acceleration is rotated back.
        """
        phi = state_world[2]
        # The state stores vx/vy in world axes, but the force balance uses body axes.
        velocity_body = world_to_body(state_world[3:5], phi)  # Rotate by -phi: [u, v].
        twist_body = np.r_[velocity_body, state_world[5]]  # [u, v, omega].

        # Coriolis/rotating-frame term for body coordinates: [-m*omega*v, m*omega*u, 0].
        coriolis = self.coriolis(twist_body, self.mass)
        # M_eff*q_dot = motor wrench + external wrench - Coriolis.
        generalized_acceleration = self.G @ current + self.mass_inverse @ (wrench_body - coriolis)

        # q_dot's first two entries are u_dot/v_dot, not physical acceleration.
        # Correct for rotating axes, then rotate the physical acceleration into world axes.
        acceleration_body = np.array([
            generalized_acceleration[0] - state_world[5] * velocity_body[1],
            generalized_acceleration[1] + state_world[5] * velocity_body[0],
        ])
        acceleration_world = body_to_world(acceleration_body, phi)
        # Required drive traction is the contact force needed to satisfy rolling
        # and wheel torque balance; MPC limits it by static friction capacity.
        # Required drive traction follows from wheel torque balance and no-slip:
        # J_w*theta_ddot = K_t*Iq + R_w*F_d, theta_ddot = -H.T*q_dot/R_w.
        traction = (
            -(MOTOR_KT / self.wheel_radius) * current
            - (self.wheel_inertia / self.wheel_radius**2) * (self.H.T @ generalized_acceleration)
        )
        theta_ddot = -(self.H.T @ generalized_acceleration) / self.wheel_radius

        # Derivative order matches state order: pose rates, world acceleration,
        # yaw acceleration, then the four wheel angular accelerations.
        derivative_world = np.r_[state_world[3:5], state_world[5], acceleration_world, generalized_acceleration[2], theta_ddot]
        return derivative_world, traction, generalized_acceleration, velocity_body

    def wheel_dynamics(self, current, traction, generalized_acceleration, state_world):
        """Return integrated wheel speeds and the existing no-slip acceleration relation."""
        theta_dot = state_world[6:10].copy()
        theta_ddot = -(self.H.T @ generalized_acceleration) / self.wheel_radius
        motor_torque = MOTOR_KT * current
        signed_load_torque = self.wheel_radius * traction
        torque_residual = self.wheel_inertia * theta_ddot - (motor_torque + signed_load_torque)
        return theta_dot, theta_ddot, motor_torque, signed_load_torque, torque_residual

    def discrete_map(self, state, current, wrench_body):
        """Advance all ten states together with classical fourth-order Runge-Kutta.

        Current and external wrench are held constant over this timestep. The
        plant derivative is evaluated at each RK4 intermediate state.
        """
        dt = self.dt
        k1 = self.physical_dynamics(state, current, wrench_body)[0]
        k2 = self.physical_dynamics(state + 0.5 * dt * k1, current, wrench_body)[0]
        k3 = self.physical_dynamics(state + 0.5 * dt * k2, current, wrench_body)[0]
        k4 = self.physical_dynamics(state + dt * k3, current, wrench_body)[0]
        return state + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

    def _linearize(self, function, state0, current0, wrench0):
        """Finite-difference A/B/E around the nominal state, current, and wrench.

        The physical dynamics remain nonlinear, but this local affine model is
        what each convex MPC subproblem predicts over its short horizon.
        """
        value0 = np.asarray(function(state0, current0, wrench0), dtype=float)
        jacobians = [np.zeros((value0.size, state0.size)),
                     np.zeros((value0.size, N_WHEELS)),
                     np.zeros((value0.size, 3))]
        epsilon = 1e-5
        for block, width in enumerate((state0.size, N_WHEELS, 3)):
            for column in range(width):
                plus = [state0.copy(), current0.copy(), wrench0.copy()]
                minus = [state0.copy(), current0.copy(), wrench0.copy()]
                plus[block][column] += epsilon
                minus[block][column] -= epsilon
                jacobians[block][:, column] = (function(*plus) - function(*minus)) / (2.0 * epsilon)
        affine = value0 - jacobians[0] @ state0 - jacobians[1] @ current0 - jacobians[2] @ wrench0
        return (*jacobians, affine)

    def linear_model(self, time_index=None, wrench_body=None):
        """Locally linearize world-state dynamics, body wrench, acceleration, and traction."""
        time_index = self.index if time_index is None else int(time_index)
        time_s = time_index * self.dt
        wrench0 = np.zeros(3) if wrench_body is None else np.asarray(wrench_body, dtype=float)
        # The axial friction law is nonlinear in wheel-frame slip. For the
        # current MPC stage, linearize at the measured plant state so a large
        # tracking error is not extrapolated from a zero-slip reference.
        state0 = (
            self.state.copy() if time_index == self.index
            else self.reference_state(time_s)
        )
        eps = max(1e-4, self.dt * 0.25)
        twist_ref_dot = (
            self.reference_state(time_s + eps)[3:6]
            - self.reference_state(time_s - eps)[3:6]
        ) / (2.0 * eps)
        velocity_body = world_to_body(state0[3:5], state0[2])
        # twist_ref_dot[:2] is world-frame acceleration because the stored
        # reference velocities are world-frame. Convert it to body axes first.
        # Since a_body = [u_dot - omega*v, v_dot + omega*u], recover the
        # body-twist derivative required by the wheel model as:
        # [u_dot, v_dot] = a_body + [omega*v, -omega*u].
        acceleration_body_ref = world_to_body(twist_ref_dot[:2], state0[2])
        qdot_ref = np.r_[
            acceleration_body_ref + np.array([
                state0[5] * velocity_body[1], -state0[5] * velocity_body[0],
            ]),
            twist_ref_dot[2],
        ]
        current0 = np.linalg.pinv(self.G) @ (
            qdot_ref - self.mass_inverse @ (wrench0 - self.coriolis(np.r_[velocity_body, state0[5]], self.mass))
        )

        A, B, E, c = self._linearize(self.discrete_map, state0, current0, wrench0)
        acceleration_map = self._linearize(
            lambda state, current, wrench: self.physical_dynamics(state, current, wrench)[0][3:5],
            state0, current0, wrench0,
        )
        traction_map = self._linearize(
            lambda state, current, wrench: self.physical_dynamics(state, current, wrench)[1],
            state0, current0, wrench0,
        )
        return {
            "A": A, "B": B, "E": E, "c": c,
            "acceleration_x": acceleration_map[0], "acceleration_u": acceleration_map[1],
            "acceleration_w": acceleration_map[2], "acceleration_c": acceleration_map[3],
            "traction_x": traction_map[0], "traction_u": traction_map[1],
            "traction_w": traction_map[2], "traction_c": traction_map[3],
            "nominal_state": state0, "nominal_current": current0,
        }

    def candidate_feasible(self, scale, request, state, wrench, model):
        current = scale * request
        if model is None:
            # Check PID's candidate against the actual nonlinear equations.
            derivative, traction, _, _ = self.physical_dynamics(state, current, wrench)
            next_state = self.discrete_map(state, current, wrench)
            acceleration = derivative[3:5]
        else:
            next_state = model["A"] @ state + model["B"] @ current + model["E"] @ wrench + model["c"]
            acceleration = model["acceleration_x"] @ state + model["acceleration_u"] @ current + model["acceleration_w"] @ wrench + model["acceleration_c"]
            traction = model["traction_x"] @ state + model["traction_u"] @ current + model["traction_w"] @ wrench + model["traction_c"]
        tolerance = MPC_CONSTRAINT_TOL
        return (
            # Current is an actuator bound, so enforce it exactly even while
            # allowing solver-scale tolerance on the other modeled constraints.
            np.max(np.abs(current)) <= self.current_limit
            and (ALLOW_DRIVE_TRACTION_OVERLOAD or np.all(np.abs(traction) <= self.wheel_traction_limits + tolerance))
            and np.max(self.facets @ acceleration) <= self.acceleration_limit_poly + tolerance
            and np.max(self.facets @ next_state[3:5]) <= self.velocity_limit_poly + tolerance
            and abs(next_state[5]) <= self.max_yaw_rate + tolerance
        )

    def largest_feasible_current_scale(self, request, state, wrench, model):
        # Enforce the box current limit analytically before nonlinear feasibility checks.
        # This avoids dozens of expensive plant integrations when current is the only
        # active constraint (the common PID saturation case).
        peak_request = float(np.max(np.abs(request)))
        current_limited_scale = (
            min(1.0, self.current_limit / peak_request) if peak_request > 0.0 else 1.0
        )
        if self.candidate_feasible(current_limited_scale, request, state, wrench, model):
            return current_limited_scale, True
        if not self.candidate_feasible(0.0, request, state, wrench, model):
            return 0.0, False
        low, high = 0.0, current_limited_scale
        # Every low endpoint is checked against the same plant map used by the
        # simulation. Fourteen bisections retain a feasible command while limiting
        # current-scale resolution to 1/16384 of the requested vector.
        for _ in range(CURRENT_FEASIBILITY_BISECTION_STEPS):
            middle = 0.5 * (low + high)
            if self.candidate_feasible(middle, request, state, wrench, model):
                low = middle
            else:
                high = middle
        return low, True

    def project_current_to_feasible(self, request, state, wrench, model):
        """Project an inaccurate MPC first action onto one-step plant limits.

        This is a small convex QP. It also handles boundary states where zero
        current is not a feasible fallback because the robot must actively brake.
        """
        import cvxpy as cp

        current = cp.Variable(N_WHEELS)
        next_state = (
            model["A"] @ state + model["B"] @ current
            + model["E"] @ wrench + model["c"]
        )
        acceleration = (
            model["acceleration_x"] @ state
            + model["acceleration_u"] @ current
            + model["acceleration_w"] @ wrench
            + model["acceleration_c"]
        )
        traction = (
            model["traction_x"] @ state
            + model["traction_u"] @ current
            + model["traction_w"] @ wrench
            + model["traction_c"]
        )
        constraints = [
            current <= self.current_limit,
            -current <= self.current_limit,
            traction <= self.wheel_traction_limits,
            -traction <= self.wheel_traction_limits,
            self.facets @ acceleration <= self.acceleration_limit_poly,
            self.facets @ next_state[3:5] <= self.velocity_limit_poly,
            next_state[5] <= self.max_yaw_rate,
            -next_state[5] <= self.max_yaw_rate,
        ]
        problem = cp.Problem(cp.Minimize(cp.sum_squares(current - request)), constraints)
        problem.solve(
            solver=cp.OSQP, warm_start=True, verbose=False,
            max_iter=100000, eps_abs=1e-6, eps_rel=1e-6,
        )
        if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE) or current.value is None:
            raise RuntimeError(
                f"No feasible one-step current projection at t={self.index * self.dt:.3f}s "
                f"(QP status: {problem.status})."
            )
        projected = np.asarray(current.value, dtype=float).copy()
        if not self.candidate_feasible(1.0, projected, state, wrench, model):
            raise RuntimeError(
                f"One-step current projection still violates plant limits at "
                f"t={self.index * self.dt:.3f}s (QP status: {problem.status})."
            )
        return projected, problem.status

    def step(self, controller):
        time_s = self.index * self.dt
        wrench = np.asarray(self.disturbance_fn(time_s, self.state.copy()), dtype=float)
        # Sample once per step. Faster motion produces a stronger random load
        # redistribution, while the callback keeps the model easy to disable.
        if self.wheel_load_fn is None:
            self.normal_loads = np.full(N_WHEELS, F_N, dtype=float)
        else:
            speed = float(np.linalg.norm(self.state[3:5]))
            self.normal_loads = np.asarray(self.wheel_load_fn(speed, self.dt), dtype=float)
            if self.normal_loads.shape != (N_WHEELS,) or np.any(self.normal_loads < 0.0):
                raise ValueError("wheel_load_fn must return four nonnegative normal loads")
            if not np.isclose(np.sum(self.normal_loads), ROBOT_MASS * GRAVITY, atol=1e-8):
                raise ValueError("wheel_load_fn must conserve total robot weight")
        self.wheel_traction_limits = MU_D_STAT * self.normal_loads
        if wrench.shape != (3,):
            raise ValueError("disturbance_fn must return body-frame [Fx, Fy, torque]")
        # PID uses the exact nonlinear plant directly. Finite-difference
        # linearization is needed only by MPC and is costly for this model.
        model = (
            self.linear_model(self.index, wrench)
            if isinstance(controller, ConvexMPCController) else None
        )
        decision = controller.compute(self, wrench, model)
        current = np.asarray(decision["current"], dtype=float)
        before = self.state.copy()
        derivative, traction, generalized_acceleration, velocity_body = self.physical_dynamics(before, current, wrench)
        theta_dot, theta_ddot, motor_torque, signed_load_torque, wheel_residual = self.wheel_dynamics(
            current, traction, generalized_acceleration, before
        )
        after = (
            model["A"] @ before + model["B"] @ current
            + model["E"] @ wrench + model["c"]
            if model is not None else self.discrete_map(before, current, wrench)
        )
        margins = {
            "current_A": self.current_limit - np.abs(current),
            "traction_N": self.wheel_traction_limits - np.abs(traction),
            "acceleration_facets": self.acceleration_limit_poly - self.facets @ derivative[3:5],
            "velocity_facets": self.velocity_limit_poly - self.facets @ after[3:5],
            "yaw_rate_rad_s": np.array([self.max_yaw_rate - abs(after[5])]),
        }
        clearance_records = obstacle_clearance_report(after[:2], time_s + self.dt)
        obstacle_margins = np.array(
            [record["margin_m"] for record in clearance_records] or [np.inf], dtype=float
        )
        # In overload-diagnostic mode, negative traction margin marks predicted
        # breakaway; it is reported but is not treated as a solver/plant failure.
        checked_margins = {key: value for key, value in margins.items()
                           if not (ALLOW_DRIVE_TRACTION_OVERLOAD and key == "traction_N")}
        minimum_margin = min(float(np.min(values)) for values in checked_margins.values())
        minimum_obstacle_margin = float(np.min(obstacle_margins))
        collision = minimum_obstacle_margin < 0.0
        residual = self.body_mass @ generalized_acceleration + self.coriolis(np.r_[velocity_body, before[5]], self.mass) - self.H @ traction - wrench
        position_ref, velocity_ref, acceleration_ref = trajectory_reference(time_s)
        # PID supplies online route-projection targets; MPC keeps fixed clock targets.
        position_ref = decision.get("position_ref", position_ref)
        velocity_ref = decision.get("velocity_ref", velocity_ref)
        acceleration_ref = decision.get("acceleration_ref", acceleration_ref)
        heading_ref = decision.get("heading_ref", self.heading_ref(time_s))
        yaw_rate_ref = decision.get("yaw_rate_ref", self.yaw_rate_ref(time_s))
        snapshot = {
            "step": self.index, "time": time_s, "dt": self.dt,
            "controller": type(controller).__name__, "state_before": before,
            "state_after": after.copy(),
            "reference": {"position": position_ref, "velocity": velocity_ref,
                          "acceleration": acceleration_ref, "heading": heading_ref,
                          "yaw_rate": yaw_rate_ref},
            "linear_model": model, "disturbance_wrench_body": wrench.copy(),
            "decision": decision, "current": current.copy(),
            "state_derivative": derivative, "velocity_body": velocity_body,
            "wheel_speed": theta_dot.copy(), "wheel_speed_rolling": theta_dot.copy(),
            "wheel_angular_acceleration": theta_ddot.copy(),
            "motor_torque": motor_torque, "signed_load_torque": signed_load_torque,
            "wheel_torque_balance_residual": wheel_residual,
            "required_traction": traction.copy(), "drive_slip": np.zeros(N_WHEELS),
            "normal_loads_N": self.normal_loads.copy(),
            "wheel_traction_limits_N": self.wheel_traction_limits.copy(),
            "static_traction_margin": margins["traction_N"].copy(),
            "slip_onset": (np.abs(traction) > self.wheel_traction_limits + MPC_CONSTRAINT_TOL),
            "force_balance_residual": residual, "constraint_margins": margins,
            "minimum_constraint_margin": minimum_margin, "feasible": minimum_margin >= -MPC_CONSTRAINT_TOL and not collision,
            "obstacle_clearance": clearance_records,
            "minimum_obstacle_clearance_margin": minimum_obstacle_margin,
            "collision": collision,
        }
        self.state = after
        self.index += 1
        self.history.append(snapshot)
        if hasattr(controller, "observe"):
            controller.observe(snapshot)
        return snapshot

    def run(self, controller, steps):
        return [self.step(controller) for _ in range(int(steps))]


In [ ]:
class ContinuousHeadingRobotSimulator(RollingOnlyRobotSimulator):
    """Keep the reference angle on the same continuous branch as physical phi."""

    def heading_ref(self, time_s):
        raw_heading = super().heading_ref(time_s)
        if not hasattr(self, "state"):
            return raw_heading
        return float(self.state[2] + wrap_angle(raw_heading - self.state[2]))

In [ ]:
class WheelSlipRobotSimulator(ContinuousHeadingRobotSimulator):
    """Coupled chassis/wheel plant using Henry's project-defined d/a slip forces."""

    enforces_no_slip = False

    def __init__(self, *args, **kwargs):
        supplied_state = kwargs.get("initial_state")
        positional_args = list(args)
        if supplied_state is None and len(positional_args) > 1:
            supplied_state = positional_args[1]
        full_state = None
        if supplied_state is not None:
            supplied_state = np.asarray(supplied_state, dtype=float)
            if supplied_state.shape == (14,):
                full_state = supplied_state.copy()
                if len(positional_args) > 1:
                    positional_args[1] = supplied_state[:10]
                    args = tuple(positional_args)
                else:
                    kwargs["initial_state"] = supplied_state[:10]
        super().__init__(*args, **kwargs)
        if full_state is not None:
            self.state = full_state
        if self.state.shape == (10,):
            self.state = np.r_[self.state, np.zeros(N_WHEELS)]
        if self.state.shape != (14,):
            raise ValueError("wheel-slip initial_state must contain 6 robot states, 4 wheel speeds, and optionally 4 wheel angles")
        self.initial_state = self.state.copy()
        # Independent wheel rotation uses the chassis rigid-body inertia only.
        self.mass_inverse = np.linalg.inv(self.body_mass)

    def reference_state(self, time_s):
        """Return reference pose/speeds with wheel angles initialized to zero."""
        return np.r_[super().reference_state(time_s), np.zeros(N_WHEELS)]

    def _contact_wrenches(self, state_world):
        phi = state_world[2]
        velocity_body = world_to_body(state_world[3:5], phi)
        force_vectors, force_d, force_a, slip_d, slip_a = wheel_contact_outputs(
            velocity_body, state_world[5], state_world[6:10],
            wheel_angles=state_world[10:14], normal_loads=self.normal_loads
        )
        drive, axle = wheel_directions_body()
        radius = WHEEL_ROBOT_DIST * axle
        drive_wrench = self.H @ force_d
        axle_vectors = axle * force_a
        axle_wrench = np.r_[
            np.sum(axle_vectors, axis=1),
            np.sum(radius[0] * axle_vectors[1] - radius[1] * axle_vectors[0]),
        ]
        return velocity_body, force_vectors, force_d, force_a, slip_d, slip_a, drive_wrench, axle_wrench

    def physical_dynamics(self, state_world, current, wrench_body):
        """Return coupled body/wheel rates from project slip and contact forces."""
        (velocity_body, _, force_d, _, slip_d, slip_a,
         drive_wrench, axle_wrench) = self._contact_wrenches(state_world)
        twist_body = np.r_[velocity_body, state_world[5]]
        coriolis = self.coriolis(twist_body, self.mass)
        total_wrench = drive_wrench + axle_wrench + np.asarray(wrench_body, dtype=float)
        generalized_acceleration = self.mass_inverse @ (total_wrench - coriolis)
        acceleration_body = np.array([
            generalized_acceleration[0] - state_world[5] * velocity_body[1],
            generalized_acceleration[1] + state_world[5] * velocity_body[0],
        ])
        acceleration_world = body_to_world(acceleration_body, state_world[2])
        motor_torque = MOTOR_KT * np.asarray(current, dtype=float)
        theta_ddot = (motor_torque + self.wheel_radius * force_d) / self.wheel_inertia
        derivative_world = np.r_[
            state_world[3:5], state_world[5], acceleration_world,
            generalized_acceleration[2], theta_ddot, state_world[6:10],
        ]
        return derivative_world, force_d, generalized_acceleration, velocity_body

    def wheel_dynamics(self, current, traction, generalized_acceleration, state_world):
        """Return actual wheel rates and the matching motor/contact torque balance."""
        theta_dot = state_world[6:10].copy()
        theta_ddot = (MOTOR_KT * np.asarray(current, dtype=float) + self.wheel_radius * traction) / self.wheel_inertia
        motor_torque = MOTOR_KT * np.asarray(current, dtype=float)
        signed_load_torque = self.wheel_radius * traction
        residual = self.wheel_inertia * theta_ddot - motor_torque - signed_load_torque
        return theta_dot, theta_ddot, motor_torque, signed_load_torque, residual

    @staticmethod
    def _project_force_slope(slip, normal_loads, mu_static, mu_dynamic, alpha, smoothing):
        """Analytic derivative of the project's smooth Williams force law."""
        slip = np.asarray(slip, dtype=float)
        loads = np.asarray(normal_loads, dtype=float)
        decay = np.exp(-alpha * np.abs(slip))
        mu = mu_dynamic + (mu_static - mu_dynamic) * decay
        dmu = -(mu_static - mu_dynamic) * alpha * np.sign(slip) * decay
        smooth = (2.0 / np.pi) * np.arctan(smoothing * slip)
        dsmooth = (2.0 / np.pi) * smoothing / (1.0 + (smoothing * slip)**2)
        return -loads * (dmu * smooth + mu * dsmooth)

    def _velocity_rhs_and_jacobian(self, candidate, state_template, current, wrench_body):
        """Return body-twist/wheel-speed rates and their state Jacobian."""
        q = candidate[:3]
        theta_dot = candidate[3:]
        velocity_body = q[:2]
        omega = q[2]
        _, force_d, force_a, slip_d, slip_a = wheel_contact_outputs(
            velocity_body, omega, theta_dot,
            wheel_angles=state_template[10:14], normal_loads=self.normal_loads
        )
        drive, axle = wheel_directions_body()
        radius = WHEEL_ROBOT_DIST * axle
        h_drive = self.H
        h_axle = np.vstack((
            axle,
            radius[0] * axle[1] - radius[1] * axle[0],
        ))
        wrench_contact = h_drive @ force_d + h_axle @ force_a
        coriolis = self.coriolis(q, self.mass)
        qdot = self.mass_inverse @ (wrench_contact + wrench_body - coriolis)
        theta_ddot = (MOTOR_KT * current + self.wheel_radius * force_d) / self.wheel_inertia
        rhs = np.r_[qdot, theta_ddot]

        # Linear maps from body twist and wheel speed to each wheel's d/a slip.
        center_rotation = np.vstack((-radius[1], radius[0]))
        d_twist = np.column_stack((
            drive.T, np.sum(drive * center_rotation, axis=0),
        ))
        a_twist = np.column_stack((
            axle.T, np.sum(axle * center_rotation, axis=0),
        ))
        d_slip_jac = np.column_stack((d_twist, self.wheel_radius * np.eye(N_WHEELS)))
        a_slip_jac = np.column_stack((a_twist, np.zeros((N_WHEELS, N_WHEELS))))
        slope_d = self._project_force_slope(
            slip_d, self.normal_loads, MU_D_STAT,
            MU_D_DYN, ALPHA_D,
            K_SMOOTH_D,
        )
        slope_a = self._project_force_slope(
            slip_a, self.normal_loads, MU_A_STAT,
            MU_A_DYN, ALPHA_A,
            K_SMOOTH_A,
        )
        contact = wheel_contact_mask(state_template[10:14]).astype(float)
        slope_d *= contact
        slope_a *= contact
        contact_jac = (
            h_drive @ (slope_d[:, None] * d_slip_jac)
            + h_axle @ (slope_a[:, None] * a_slip_jac)
        )
        coriolis_jac = np.array([
            [0.0, -self.mass * omega, -self.mass * q[1]],
            [self.mass * omega, 0.0, self.mass * q[0]],
            [0.0, 0.0, 0.0],
        ])
        jacobian = np.zeros((3 + N_WHEELS, 3 + N_WHEELS))
        jacobian[:3] = self.mass_inverse @ (contact_jac - np.column_stack((
            coriolis_jac, np.zeros((3, N_WHEELS)),
        )))
        jacobian[3:] = (
            self.wheel_radius / self.wheel_inertia
        ) * (slope_d[:, None] * d_slip_jac)
        return rhs, jacobian

    def _implicit_velocity_step(self, state, current, wrench_body, h):
        """Solve one stiff implicit velocity/wheel-speed step with Newton updates."""
        phi = state[2]
        velocity_body = world_to_body(state[3:5], phi)
        q_before = np.r_[velocity_body, state[5]]
        start = np.r_[q_before, state[6:10]]
        guess = start.copy()
        identity = np.eye(start.size)
        state_template = np.asarray(state, dtype=float)
        converged = False
        for _ in range(12):
            stage_state = state_template.copy()
            stage_state[10:14] = state[10:14] + 0.5 * h * (state[6:10] + guess[3:])
            rhs, jacobian = self._velocity_rhs_and_jacobian(
                guess, stage_state, current, wrench_body
            )
            residual = guess - start - h * rhs
            residual_norm = np.linalg.norm(residual, ord=np.inf)
            if residual_norm < 1e-10 * (1.0 + np.linalg.norm(start, ord=np.inf)):
                converged = True
                break
            correction = np.linalg.solve(identity - h * jacobian, residual)
            # A short backtracking search protects Newton near the force-law
            # transition while keeping the local implicit solve deterministic.
            accepted = False
            for fraction in (1.0, 0.5, 0.25, 0.125, 0.0625):
                candidate = guess - fraction * correction
                candidate_state = state_template.copy()
                candidate_state[10:14] = state[10:14] + 0.5 * h * (state[6:10] + candidate[3:])
                rhs_candidate = self._velocity_rhs_and_jacobian(
                    candidate, candidate_state, current, wrench_body
                )[0]
                candidate_residual = candidate - start - h * rhs_candidate
                if np.linalg.norm(candidate_residual, ord=np.inf) < residual_norm:
                    guess = candidate
                    accepted = True
                    break
            if not accepted:
                raise RuntimeError("Implicit wheel/contact Newton update did not decrease its residual")
        if not converged:
            stage_state = state_template.copy()
            stage_state[10:14] = state[10:14] + 0.5 * h * (state[6:10] + guess[3:])
            final_rhs = self._velocity_rhs_and_jacobian(
                guess, stage_state, current, wrench_body
            )[0]
            final_residual = guess - start - h * final_rhs
            if np.linalg.norm(final_residual, ord=np.inf) >= 1e-10 * (1.0 + np.linalg.norm(start, ord=np.inf)):
                raise RuntimeError("Implicit wheel/contact Newton update did not converge")

        phi_next = phi + 0.5 * h * (q_before[2] + guess[2])
        phi_mid = 0.5 * (phi + phi_next)
        velocity_mid_body = 0.5 * (q_before[:2] + guess[:2])
        next_state = state.copy()
        next_state[:2] += body_to_world(velocity_mid_body, phi_mid) * h
        next_state[2] = phi_next
        next_state[3:5] = body_to_world(guess[:2], phi_next)
        next_state[5] = guess[2]
        next_state[6:10] = guess[3:]
        next_state[10:14] = state[10:14] + 0.5 * h * (state[6:10] + guess[3:])
        return next_state

    def discrete_map(self, state, current, wrench_body):
        """Advance the coupled chassis/wheel state with adaptive RK45."""
        result = solve_ivp(
            lambda time_s, state_world: self.physical_dynamics(
                state_world, current, wrench_body
            )[0],
            (0.0, self.dt),
            np.asarray(state, dtype=float),
            method="RK45",
            rtol=1e-5,
            atol=1e-7,
            max_step=self.dt,
        )
        if not result.success:
            raise RuntimeError(f"RK45 plant integration failed: {result.message}")
        return result.y[:, -1]

    def inverse_dynamics_current(self, desired_qdot, state_world, wrench_body):
        """Allocate wheel currents for a desired body-twist derivative.

        Drive contact forces are distributed by the pseudoinverse of the project's
        force/moment matrix. Wheel target angular acceleration uses the compatible
        no-slip acceleration relation as feed-forward; actual wheel motion remains
        free to slip under the nonlinear contact law.
        """
        velocity_body = world_to_body(state_world[3:5], state_world[2])
        twist_body = np.r_[velocity_body, state_world[5]]
        coriolis = self.coriolis(twist_body, self.mass)
        axial_wrench = self._contact_wrenches(state_world)[-1]
        required_drive_wrench = (
            self.body_mass @ np.asarray(desired_qdot, dtype=float)
            + coriolis - np.asarray(wrench_body, dtype=float) - axial_wrench
        )
        force_d_target = np.linalg.pinv(self.H) @ required_drive_wrench
        theta_ddot_target = -(self.H.T @ np.asarray(desired_qdot, dtype=float)) / self.wheel_radius
        motor_torque_target = self.wheel_inertia * theta_ddot_target - self.wheel_radius * force_d_target
        return motor_torque_target / MOTOR_KT

    def linear_model(self, time_index=None, wrench_body=None):
        """Finite-difference the nonlinear slip plant for a local MPC prediction."""
        time_index = self.index if time_index is None else int(time_index)
        time_s = time_index * self.dt
        wrench0 = np.zeros(3) if wrench_body is None else np.asarray(wrench_body, dtype=float)
        state0 = self.state.copy() if time_index == self.index else self.reference_state(time_s)

        eps = max(1e-4, self.dt * 0.25)
        position_ref, velocity_ref, acceleration_ref = trajectory_reference(time_s)
        phi_ref = state0[2]
        velocity_body_ref = world_to_body(state0[3:5], phi_ref)
        acceleration_body_ref = world_to_body(acceleration_ref, phi_ref)
        omega_ref = state0[5]
        yaw_accel_ref = (
            self.yaw_rate_ref(time_s + eps) - self.yaw_rate_ref(time_s - eps)
        ) / (2.0 * eps)
        qdot_ref = np.r_[
            acceleration_body_ref + np.array([
                omega_ref * velocity_body_ref[1], -omega_ref * velocity_body_ref[0],
            ]),
            yaw_accel_ref,
        ]
        requested_current0 = self.inverse_dynamics_current(qdot_ref, state0, wrench0)
        current0 = np.clip(requested_current0, -self.current_limit, self.current_limit)
        A, B, E, c = self._linearize(self.discrete_map, state0, current0, wrench0)

        def plant_outputs(state, current, wrench):
            derivative, force_d, _, _ = self.physical_dynamics(state, current, wrench)
            return np.r_[derivative[3:5], force_d]

        output_x, output_u, output_w, output_c = self._linearize(
            plant_outputs, state0, current0, wrench0
        )
        return {
            "A": A, "B": B, "E": E, "c": c,
            "acceleration_x": output_x[:2], "acceleration_u": output_u[:2],
            "acceleration_w": output_w[:2], "acceleration_c": output_c[:2],
            "traction_x": output_x[2:], "traction_u": output_u[2:],
            "traction_w": output_w[2:], "traction_c": output_c[2:],
            "nominal_state": state0, "nominal_current": current0,
        }

    def step(self, controller):
        snapshot = super().step(controller)
        before = snapshot["state_before"]
        current = snapshot["current"]
        wrench = snapshot["disturbance_wrench_body"]
        (velocity_body, _, force_d, force_a, slip_d, slip_a,
         drive_wrench, axle_wrench) = self._contact_wrenches(before)
        twist_body = np.r_[velocity_body, before[5]]
        coriolis = self.coriolis(twist_body, self.mass)
        _, _, generalized_acceleration, _ = self.physical_dynamics(
            before, current, wrench
        )
        theta_ddot = snapshot["wheel_angular_acceleration"]
        motor_torque = MOTOR_KT * current
        signed_load_torque = self.wheel_radius * force_d
        snapshot["wheel_torque_balance_residual"] = (
            self.wheel_inertia * theta_ddot - motor_torque - signed_load_torque
        )
        snapshot["force_balance_residual"] = (
            self.body_mass @ generalized_acceleration + coriolis
            - drive_wrench - axle_wrench - wrench
        )
        snapshot["wheel_speed"] = snapshot["state_after"][6:10].copy()
        snapshot["wheel_speed_actual_before"] = before[6:10].copy()
        snapshot["wheel_angle"] = before[10:14].copy()
        snapshot["wheel_angle_after"] = snapshot["state_after"][10:14].copy()
        snapshot["wheel_contact"] = wheel_contact_mask(before[10:14])
        snapshot["wheel_speed_rolling"] = (
            -(self.H.T @ twist_body) / self.wheel_radius
        )
        snapshot["drive_slip"] = slip_d.copy()
        snapshot["slip_a"] = slip_a.copy()
        snapshot["force_d"] = force_d.copy()
        snapshot["force_a"] = force_a.copy()
        snapshot["axial_wrench_body"] = axle_wrench.copy()
        snapshot["drive_wrench_body"] = drive_wrench.copy()
        snapshot["static_traction_margin"] = (
            self.wheel_traction_limits - np.abs(force_d)
        )
        snapshot["slip_onset"] = np.abs(slip_d) >= SLIP_THRESHOLD
        snapshot["required_traction"] = force_d.copy()
        snapshot["signed_load_torque"] = signed_load_torque.copy()
        return snapshot


## 5. PID controller and inverse dynamics

The translational PID produces a desired world-frame acceleration using position error, velocity error, integral error, and trajectory feed-forward. A wheel-force pseudoinverse allocates the corresponding drive wrench as an `Iq_ref` request, including wheel inertia and contact torque. The FOC current-loop wrapper converts that request into voltage and actual current; nonlinear plant feasibility scaling remains based on the requested current and contact laws.

Heading control is cascaded: heading error produces a yaw-rate target, and yaw-rate error produces the requested yaw acceleration. The resulting translation/yaw request is combined before current and plant-limit checks.


In [ ]:
class PositionPIDController:
    """Position PID plus a cascaded heading-to-omega PID baseline.

    The outer heading loop turns heading error into a desired yaw rate. The
    inner omega PID uses omega_ref - omega as its error, and its output is the
    yaw acceleration request that the motor/current model must realize.
    Translation and yaw requests are converted to wheel currents, then scaled
    if current or static-traction limits would be exceeded.
    """

    # Trial tuning for the current figure-eight: stronger position correction with damping.
    def __init__(self, kp=(12.0, 12.0), ki=(0.5, 0.5), kd=(6.0, 6.0),
                 kp_heading=5.0, ki_heading=0.2,
                 kp_omega=4.0, ki_omega=0.2, kd_omega=0.05):
        self.kp = np.asarray(kp, dtype=float)
        self.ki = np.asarray(ki, dtype=float)
        self.kd = np.asarray(kd, dtype=float)
        # Outer loop: heading error -> correction to the reference yaw rate.
        self.kp_heading = float(kp_heading)
        self.ki_heading = float(ki_heading)
        # Inner loop: explicit PID gains on yaw-rate error, omega_ref - omega.
        # These are starting values and still need tuning against robot data.
        self.kp_omega = float(kp_omega)
        self.ki_omega = float(ki_omega)
        self.kd_omega = float(kd_omega)
        self.integral = np.zeros(4)  # x, y, heading, and omega error integrals
        self.previous_omega_error = None
        # Progress is projected from the actual robot position each step. The
        # target therefore follows the plant instead of advancing on a clock.
        self.path_arc_progress = 0.0  # measured robot progress along the route
        self.reference_arc_progress = 0.0  # moving target progress, kept near the robot
        self.target_lead_time = 0.25  # seconds of dynamically reachable lookahead

    def _local_path_reference(self, plant):
        """Build a route target from measured position and dynamically limited speed."""
        state = plant.state
        position = state[:2]
        velocity_world = state[3:5]
        progress_tangent = np.array([
            np.interp(self.path_arc_progress, PATH_ARC, PATH_TANGENT[:, 0]),
            np.interp(self.path_arc_progress, PATH_ARC, PATH_TANGENT[:, 1]),
        ])
        speed_along_path = float(np.dot(velocity_world, progress_tangent))

        # Search a small arc-length window around the last projection. Keeping
        # this window local prevents the figure-eight crossing from jumping to
        # its other branch, and prevents a distant future point becoming target.
        back_window = 0.08
        forward_window = max(0.25, 0.15 + 0.15 * abs(speed_along_path))
        lower_arc = max(0.0, self.path_arc_progress - back_window)
        upper_arc = min(PATH_ARC[-1], self.path_arc_progress + forward_window)
        first = max(0, int(np.searchsorted(PATH_ARC, lower_arc, side="left")))
        last = min(len(PATH_ARC), int(np.searchsorted(PATH_ARC, upper_arc, side="right")))
        candidate_points = _route_points[first:last]
        if len(candidate_points) == 0:
            candidate_points = _route_points[-1:]
            first = len(PATH_ARC) - 1
        nearest = int(np.argmin(np.sum((candidate_points - position) ** 2, axis=1)))
        projected_index = first + nearest
        # Progress is monotone so noise near the crossing cannot send the robot
        # backward or switch it onto the other half of the figure-eight.
        self.path_arc_progress = max(self.path_arc_progress, float(PATH_ARC[projected_index]))
        actual_arc = self.path_arc_progress

        # The speed envelope respects route curvature and stopping distance. Its
        # command is rate-limited from measured speed, so it cannot jump ahead in
        # velocity. Look ahead one control step to avoid a zero-speed deadlock at
        # the start of the route.
        forward_speed = max(speed_along_path, 0.0)
        profile_step = max(
            0.01,
            forward_speed * plant.dt + 0.5 * TANGENTIAL_ACCEL_LIMIT * plant.dt**2,
        )
        speed_cap = float(np.interp(
            min(PATH_ARC[-1], actual_arc + profile_step), PATH_ARC, PATH_SPEED
        ))
        speed_step = TANGENTIAL_ACCEL_LIMIT * plant.dt
        if forward_speed <= speed_cap:
            speed_ref_scalar = min(speed_cap, forward_speed + speed_step)
        else:
            speed_ref_scalar = max(speed_cap, forward_speed - speed_step)
        tangential_acceleration = float(np.clip(
            (speed_ref_scalar - speed_along_path) / plant.dt,
            -TANGENTIAL_ACCEL_LIMIT, TANGENTIAL_ACCEL_LIMIT,
        ))

        # Advance a separate moving target with the speed command, but bound its
        # lead by distance the robot could cover in target_lead_time. If plant
        # dynamics slow the robot, this clamp pulls the target back toward it.
        reachable_lead = (
            forward_speed * self.target_lead_time
            + 0.5 * TANGENTIAL_ACCEL_LIMIT * self.target_lead_time**2
        )
        max_reference_arc = min(PATH_ARC[-1], actual_arc + reachable_lead)
        next_reference_arc = self.reference_arc_progress + speed_ref_scalar * plant.dt
        self.reference_arc_progress = float(np.clip(
            next_reference_arc, actual_arc, max_reference_arc
        ))
        arc = self.reference_arc_progress

        position_ref = np.array([
            np.interp(arc, PATH_ARC, _route_points[:, 0]),
            np.interp(arc, PATH_ARC, _route_points[:, 1]),
        ])
        tangent = np.array([
            np.interp(arc, PATH_ARC, PATH_TANGENT[:, 0]),
            np.interp(arc, PATH_ARC, PATH_TANGENT[:, 1]),
        ])
        tangent /= max(np.linalg.norm(tangent), 1e-9)
        normal = np.array([-tangent[1], tangent[0]])
        curvature = float(np.interp(arc, PATH_ARC, PATH_CURVATURE))
        velocity_ref = speed_ref_scalar * tangent
        acceleration_ref = (
            tangential_acceleration * tangent
            + speed_ref_scalar**2 * curvature * normal
        )
        heading_ref = float(np.arctan2(tangent[1], tangent[0]))
        yaw_rate_ref = curvature * speed_ref_scalar
        return position_ref, velocity_ref, acceleration_ref, heading_ref, yaw_rate_ref

    def compute(self, plant, wrench_body, model):
        time_s = plant.index * plant.dt
        (position_ref, velocity_ref, acceleration_ref,
         heading_ref, yaw_rate_ref) = self._local_path_reference(plant)
        state = plant.state
        position_error = position_ref - state[:2]
        velocity_error = velocity_ref - state[3:5]
        # P position + D velocity + I accumulated position error, plus route
        # acceleration feed-forward. Repulsion is a PID heuristic, not a guarantee.
        acceleration_world = (
            acceleration_ref + self.kp * position_error
            + self.ki * self.integral[:2] + self.kd * velocity_error
            + pid_obstacle_repulsion(state[:2], time_s)
        )
        heading_error = wrap_angle(heading_ref - state[2])
        # Outer loop: convert local path-heading error into a desired yaw rate.
        omega_ref = yaw_rate_ref + self.kp_heading * heading_error
        omega_ref += self.ki_heading * self.integral[2]
        omega_ref = float(np.clip(omega_ref, -MAX_YAW_RATE, MAX_YAW_RATE))
        omega_error = omega_ref - state[5]

        # Inner loop: PID error is omega_ref - omega. Its output is a yaw
        # acceleration request because wheel torque changes the body's omega
        # through the rotational dynamics. The derivative is taken on the omega
        # error itself; no wheel-angle signal is used to form this command.
        omega_error_rate = 0.0
        if self.previous_omega_error is not None:
            omega_error_rate = (omega_error - self.previous_omega_error) / plant.dt
        yaw_acceleration = (
            self.kp_omega * omega_error
            + self.ki_omega * self.integral[3]
            + self.kd_omega * omega_error_rate
        )
        velocity_body = world_to_body(state[3:5], state[2])
        acceleration_body = world_to_body(acceleration_world, state[2])
        omega = state[5]
        generalized_acceleration = np.array([
            acceleration_body[0] + omega * velocity_body[1],
            acceleration_body[1] - omega * velocity_body[0],
            yaw_acceleration,
        ])
        coriolis = plant.coriolis(np.r_[velocity_body, omega], plant.mass)
        # Allocate desired chassis wrench across the actual wheel/contact model.
        # The feed-forward wheel acceleration is kinematic, but actual wheel
        # speeds remain independent states and can depart from rolling under slip.
        requested_current = plant.inverse_dynamics_current(
            generalized_acceleration, state, wrench_body
        )
        scale, feasible = plant.largest_feasible_current_scale(
            requested_current, state, wrench_body, model
        )
        return {
            "current": scale * requested_current,
            "requested_current": requested_current,
            "tracking_error": np.r_[position_error, heading_error, omega_error],
            "omega_ref": omega_ref,
            "omega_error": omega_error,
            # Save the actual online targets so the plots show what PID followed.
            "position_ref": position_ref.copy(),
            "velocity_ref": velocity_ref.copy(),
            "acceleration_ref": acceleration_ref.copy(),
            "heading_ref": heading_ref,
            "yaw_rate_ref": yaw_rate_ref,
            "path_arc_progress": self.path_arc_progress,
            "reference_arc_progress": self.reference_arc_progress,
            "saturation_scale": scale,
            "feasible_ray": feasible,
            "freeze_integrator": scale < 1.0 - 1e-9,
        }

    def observe(self, snapshot):
        if not snapshot["decision"]["freeze_integrator"]:
            self.integral += snapshot["dt"] * snapshot["decision"]["tracking_error"]
        self.integral = np.clip(self.integral, -1.5, 1.5)
        # Save the last omega error for the inner PID's derivative term.
        self.previous_omega_error = float(snapshot["decision"]["omega_error"])


In [ ]:
class FocCurrentLoopController:
    """Wrap a robot-level current-command controller with a simplified FOC loop."""
    def __init__(self, outer_controller, q_voltage_limit=FOC_Q_AXIS_VOLTAGE_LIMIT,
                 current_bandwidth_hz=FOC_CURRENT_LOOP_BANDWIDTH_HZ):
        self.outer_controller = outer_controller
        self.q_voltage_limit = float(q_voltage_limit)
        self.current_bandwidth_hz = float(current_bandwidth_hz)
        if self.q_voltage_limit <= 0.0 or self.current_bandwidth_hz <= 0.0:
            raise ValueError("FOC voltage limit and current-loop bandwidth must be positive")
        self.current = np.zeros(N_WHEELS, dtype=float)
        self.integral_error = np.zeros(N_WHEELS, dtype=float)
        wc = 2.0 * np.pi * self.current_bandwidth_hz
        self.kp_voltage_per_amp = MOTOR_INDUCTANCE * wc
        self.ki_voltage_per_amp_second = MOTOR_RESISTANCE * wc

    def __getattr__(self, name):
        # Keep the wrapped outer controller's progress and telemetry attributes visible.
        outer = self.__dict__.get("outer_controller")
        if outer is None:
            raise AttributeError(name)
        return getattr(outer, name)

    def compute(self, plant, wrench_body, model):
        decision = self.outer_controller.compute(plant, wrench_body, model)
        iq_reference = np.clip(np.asarray(decision["current"], dtype=float),
                               -plant.current_limit, plant.current_limit)
        if iq_reference.shape != (N_WHEELS,):
            raise ValueError("current reference must contain one Iq command per wheel")
        omega = np.asarray(plant.state[6:10], dtype=float).copy()
        dt = float(plant.dt)
        n = max(1, int(np.ceil(dt / FOC_ELECTRICAL_MAX_STEP)))
        h = dt / n
        tau_e = MOTOR_INDUCTANCE / MOTOR_RESISTANCE
        decay = float(np.exp(-h / tau_e))
        current_integral = np.zeros(N_WHEELS)
        voltage_integral = np.zeros(N_WHEELS)
        saturation_count = np.zeros(N_WHEELS)
        for _ in range(n):
            error = iq_reference - self.current
            v_raw = (MOTOR_RESISTANCE * iq_reference + MOTOR_KE * omega
                     + self.kp_voltage_per_amp * error
                     + self.ki_voltage_per_amp_second * self.integral_error)
            vq = np.clip(v_raw, -self.q_voltage_limit, self.q_voltage_limit)
            saturated = np.abs(v_raw - vq) > 1e-12
            i_eq = (vq - MOTOR_KE * omega) / MOTOR_RESISTANCE
            i0 = self.current.copy()
            i1 = np.clip(i_eq + (i0 - i_eq) * decay,
                         -plant.current_limit, plant.current_limit)
            i_avg = i_eq + (i0 - i_eq) * (tau_e / h) * (1.0 - decay)
            i_avg = np.clip(i_avg, -plant.current_limit, plant.current_limit)
            current_integral += i_avg * h
            voltage_integral += vq * h
            saturation_count += saturated.astype(float)
            unwind = ((v_raw > self.q_voltage_limit) & (error < 0.0)) | (
                (v_raw < -self.q_voltage_limit) & (error > 0.0))
            self.integral_error = np.where((~saturated) | unwind,
                                           self.integral_error + error * h,
                                           self.integral_error)
            self.current = i1
        iq_average = current_integral / dt
        decision["commanded_current"] = iq_reference.copy()
        decision["current"] = iq_average
        decision["requested_current"] = np.asarray(
            decision.get("requested_current", iq_reference), dtype=float)
        decision["iq_actual_average_A"] = iq_average.copy()
        decision["iq_actual_end_A"] = self.current.copy()
        decision["vq_average_V"] = voltage_integral / dt
        decision["voltage_saturation_fraction"] = saturation_count / n
        decision["electrical_substeps"] = n
        return decision

    def observe(self, snapshot):
        if hasattr(self.outer_controller, "observe"):
            self.outer_controller.observe(snapshot)


## 6. Convex MPC: prediction model, objective, and constraints

The ideal-current MPC implementation is retained here only as reference. It does not include the FOC current state or voltage limits, so this FOC notebook runs the PID path by default. An FOC-aware MPC would need to include actual motor current and voltage. At each stage, finite differences of the nonlinear, implicitly integrated slip-plant map form

$$
\mathbf x_{k+1}=A_k\mathbf x_k+B_k\mathbf I_{q,k}+E_k\mathbf w_k+\mathbf c_k.
$$

The affine offset makes the local model match the nonlinear map at its nominal point. The quadratic objective penalizes tracking error, integral error, acceleration-reference error, current magnitude, current slew, and terminal error. Linear constraints enforce current, computed contact-force capacity, acceleration, speed, yaw-rate, integral, and obstacle limits. The QP is a local approximation; the first action is checked against the nonlinear plant map before application.

Only the first current vector is applied. The optimization is solved again at the next sample, which is the receding-horizon loop.


In [ ]:
class ConvexMPCController:
    """Receding-horizon convex QP over wheel currents and predicted robot states.

    Position, velocity, and heading references are fixed data at each stage.
    Integral error is predicted controller memory, not an extra plant state.
    """

    def __init__(self, horizon=20, weights=None):
        self.horizon = int(horizon)
        if self.horizon < 1:
            raise ValueError("horizon must be positive")
        self.integral = np.zeros(3)
        self.planned_integrals = None
        self.weights = weights or {
            "position": 60.0, "velocity": 8.0, "heading": 8.0,
            "integral": 0.05, "integral_slack": 1000.0,
            "acceleration": 2.0, "current": 0.02, "slew": 0.15,
            "terminal_position": 120.0, "terminal_velocity": 16.0,
            "terminal_heading": 20.0,
        }

    def _continuous_heading_reference(self, plant, time_s):
        return plant.state[2] + wrap_angle(plant.heading_ref(time_s) - plant.state[2])

    def compute(self, plant, wrench_body, first_model):
        import cvxpy as cp

        requested_horizon = self.horizon
        horizon = self.horizon
        # Decision variables: all physical states (including wheel speeds and
        # angles) at N+1 samples, PID integral memory, and four wheel currents.
        # Wheel currents have N columns because each current advances one interval.
        states = cp.Variable((plant.state.size, horizon + 1))
        integrals = cp.Variable((3, horizon + 1))
        integral_slack = cp.Variable((3, horizon), nonneg=True)
        # Obstacle clearance is softened so an already unsafe state or a changed
        # waypoint cannot make the whole QP infeasible for one short time step.
        # A large penalty makes the optimizer satisfy clearance whenever feasible.
        obstacle_slacks = []
        currents = cp.Variable((N_WHEELS, horizon))
        constraints = [states[:, 0] == plant.state, integrals[:, 0] == self.integral]
        objective = 0

        for offset in range(horizon):
            index = plant.index + offset
            next_time = (index + 1) * plant.dt
            model = first_model if offset == 0 else plant.linear_model(index, np.zeros(3))
            # These are desired targets at this future time, not variables. For a
            # receive-pass event the terminal target is the goal with zero velocity.
            position_ref, velocity_ref, acceleration_ref = trajectory_reference(next_time)
            heading_ref = self._continuous_heading_reference(plant, next_time)
            stage_wrench = wrench_body if offset == 0 else np.zeros(3)
            current = currents[:, offset]
            # The local affine maps express acceleration and required wheel
            # traction as functions of predicted state and candidate currents.
            acceleration = (
                model["acceleration_x"] @ states[:, offset]
                + model["acceleration_u"] @ current
                + model["acceleration_w"] @ stage_wrench
                + model["acceleration_c"]
            )
            traction = (
                model["traction_x"] @ states[:, offset]
                + model["traction_u"] @ current
                + model["traction_w"] @ stage_wrench
                + model["traction_c"]
            )
            # Matrix state update: x[k+1] = A*x[k] + B*Iq[k] + E*w + c.
            # The next two equalities accumulate position and heading error.
            constraints += [
                states[:, offset + 1] == model["A"] @ states[:, offset]
                    + model["B"] @ current + model["E"] @ stage_wrench + model["c"],
                integrals[:2, offset + 1] == integrals[:2, offset]
                    + plant.dt * (position_ref - states[:2, offset]),
                integrals[2, offset + 1] == integrals[2, offset]
                    + plant.dt * (heading_ref - states[2, offset]),
                integrals[:, offset + 1] <= 1.5 + integral_slack[:, offset],
                -integrals[:, offset + 1] <= 1.5 + integral_slack[:, offset],
                current <= plant.current_limit,
                -current <= plant.current_limit,
                # The bounded project contact-force law stays within this capacity.
                *([] if ALLOW_DRIVE_TRACTION_OVERLOAD else [
                    traction <= plant.wheel_traction_limits,
                    -traction <= plant.wheel_traction_limits,
                ]),
                # Facets make polygonal (therefore linear) bounds for acceleration
                # and speed instead of nonlinear Euclidean-norm constraints.
                plant.facets @ acceleration <= plant.acceleration_limit_poly,
                plant.facets @ states[3:5, offset + 1] <= plant.velocity_limit_poly,
                cp.abs(states[5, offset + 1]) <= plant.max_yaw_rate,
            ]
            # Every obstacle has one preselected safe side. Its separating line
            # is fixed data for this solve, so the added constraint is affine.
            # The obstacle side and separating hyperplane are constants for this
            # QP stage, selected from the nominal reference and predicted obstacle pose.
            for _, obstacle_vertices, obstacle_clearance in obstacle_polygons_at(next_time):
                normal, bound = obstacle_safe_halfspace(
                    position_ref, obstacle_vertices, clearance=obstacle_clearance
                )
                clearance_slack = cp.Variable(nonneg=True)
                obstacle_slacks.append(clearance_slack)
                constraints.append(
                    normal @ states[:2, offset + 1] + clearance_slack >= bound
                )
                # Squared nonnegative slack keeps this a convex QP. It allows a
                # temporary clearance shortfall only when the hard plant limits
                # prevent clearing the obstacle at this sample.
                objective += 1e4 * cp.square(clearance_slack)
            # Quadratic tracking costs target pose/velocity; current and slew
            # penalties regularize effort without changing convexity.
            objective += self.weights["position"] * cp.sum_squares(states[:2, offset + 1] - position_ref)
            objective += self.weights["velocity"] * cp.sum_squares(states[3:5, offset + 1] - velocity_ref)
            objective += self.weights["heading"] * cp.square(states[2, offset + 1] - heading_ref)
            objective += self.weights["integral"] * cp.sum_squares(integrals[:, offset + 1])
            objective += self.weights["integral_slack"] * cp.sum_squares(integral_slack[:, offset])
            objective += self.weights["acceleration"] * cp.sum_squares(acceleration - acceleration_ref)
            objective += self.weights["current"] * cp.sum_squares(current)
            previous = plant.history[-1]["current"] if offset == 0 and plant.history else np.zeros(N_WHEELS)
            if offset > 0:
                previous = currents[:, offset - 1]
            objective += self.weights["slew"] * cp.sum_squares(current - previous)

        terminal_time = (plant.index + horizon) * plant.dt
        terminal_position, terminal_velocity, _ = trajectory_reference(terminal_time)
        terminal_heading = self._continuous_heading_reference(plant, terminal_time)
        objective += self.weights["terminal_position"] * cp.sum_squares(states[:2, -1] - terminal_position)
        objective += self.weights["terminal_velocity"] * cp.sum_squares(states[3:5, -1] - terminal_velocity)
        objective += self.weights["terminal_heading"] * cp.square(states[2, -1] - terminal_heading)
        # Apply only the first current; measure the new state and solve again
        # on the next step (the receding-horizon control loop).
        # Solve the convex QP, apply only the first current vector, then replan
        # at the next sample using the latest state (receding-horizon control).
        problem = cp.Problem(cp.Minimize(objective), constraints)
        problem.solve(
            solver=cp.OSQP, warm_start=True, verbose=False,
            max_iter=50000, eps_abs=MPC_CONSTRAINT_TOL, eps_rel=MPC_CONSTRAINT_TOL,
        )
        if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            if horizon > 1:
                shorter = ConvexMPCController(horizon=horizon - 1, weights=self.weights)
                shorter.integral = self.integral.copy()
                decision = shorter.compute(plant, wrench_body, first_model)
                self.planned_integrals = shorter.planned_integrals
                decision["requested_horizon"] = requested_horizon
                return decision
            raise RuntimeError(
                f"MPC QP did not solve at t={plant.index * plant.dt:.3f}s "
                f"after reducing to horizon 1 (solver status: {problem.status}). "
                "A status of 'user_limit' means OSQP reached its iteration limit, "
                "not that the constraints were proven infeasible."
            )
        requested_current = currents.value[:, 0].copy()
        # OSQP can return a numerically inaccurate iterate. Check the first
        # action against the plant limits and scale it toward zero if needed.
        if plant.candidate_feasible(
            1.0, requested_current, plant.state, wrench_body, first_model
        ):
            applied_current = requested_current
            safety_scale = 1.0
            safety_projection_status = None
        elif plant.candidate_feasible(
            0.0, requested_current, plant.state, wrench_body, first_model
        ):
            safety_scale, _ = plant.largest_feasible_current_scale(
                requested_current, plant.state, wrench_body, first_model
            )
            applied_current = safety_scale * requested_current
            safety_projection_status = None
        else:
            # Scaling toward zero fails when the robot must apply braking current
            # to recover a saturated speed/yaw state, so solve a small projection QP.
            applied_current, safety_projection_status = plant.project_current_to_feasible(
                requested_current, plant.state, wrench_body, first_model
            )
            safety_scale = None
        # Enforce the hard actuator cap exactly despite OSQP feasibility tolerance.
        hard_current_limit = min(float(plant.current_limit), float(IQ_LIMIT))
        applied_current = np.clip(applied_current, -hard_current_limit, hard_current_limit)
        self.planned_integrals = integrals.value[:, 1].copy()
        return {
            "current": applied_current,
            "requested_current": requested_current,
            "first_step_safety_scale": safety_scale,
            "first_step_safety_projection_status": safety_projection_status,
            "qp_status": problem.status,
            "qp_objective": float(problem.value),
            "planned_currents": currents.value.copy(),
            "planned_states": states.value.copy(),
            "planned_integrals": integrals.value.copy(),
            "planned_obstacle_slacks": np.asarray(
                [slack.value for slack in obstacle_slacks], dtype=float
            ),
            "effective_horizon": horizon,
            "requested_horizon": requested_horizon,
            "integral_slack": integral_slack.value.copy(),
        }

    def observe(self, snapshot):
        if self.planned_integrals is not None:
            self.integral = np.clip(self.planned_integrals, -1.5, 1.5)


## 7. Run the simulation and inspect diagnostics

The next cell runs a short controller preview, executes the selected simulation, checks model/constraint outputs, and builds the animation and diagnostic plots. The animation shows the physical pose; its drawn outline is for visualization, while collision checking uses the simulator's separate clearance geometry.


### Summary dashboard

The compact 2×2 summary below takes its presentation layout from Dana’s [second robot model notebook](dana-models/robot2_dynamics_motor_friction_lqr_plan_comparison_pid_explained.ipynb): trajectory and tracking error beside actuator and contact diagnostics. Here, the actuator panels show commanded versus actual q-axis current and q-axis voltage; the detailed plots below retain this project’s d/a slip and force diagnostics. Only the plotting organization is borrowed; the project model equations and geometry remain in use.

In [ ]:
def make_selected_controller():
    if CONTROLLER_MODE.upper() == "PID":
        return FocCurrentLoopController(PositionPIDController())
    if CONTROLLER_MODE.upper() == "MPC":
        raise NotImplementedError(
            "The FOC current state is not included in this notebook's MPC prediction. "
            "Use PID here; ideal-current MPC remains in the original notebook."
        )
    raise ValueError("CONTROLLER_MODE must be 'PID' for this FOC-current-loop notebook")


preview_plant = WheelSlipRobotSimulator()
preview_controller = make_selected_controller()
for _ in range(3):
    snapshot = preview_plant.step(preview_controller)
    print(
        f"{CONTROLLER_MODE.upper()} k={snapshot['step']} t={snapshot['time']:.2f}s "
        f"p={np.round(snapshot['state_after'][:2], 4)} "
        f"i={np.round(snapshot['current'], 4)} "
        f"min_margin={snapshot['minimum_constraint_margin']:.4f}"
    )
if snapshot["linear_model"] is not None:
    print("Model shapes A/B/E:", snapshot["linear_model"]["A"].shape,
          snapshot["linear_model"]["B"].shape, snapshot["linear_model"]["E"].shape)
else:
    print("PID uses direct nonlinear plant dynamics; skipped MPC linearization.")

GOAL_HOLD_TIME = 1.0  # Require a full second at rest before declaring arrival.
plant = WheelSlipRobotSimulator()
run_controller = make_selected_controller()
if CONTROLLER_MODE.upper() == "PID":
    # PID follows actual route progress, so end the run when the plant reaches
    # the endpoint and settles. A generous cap protects against endless runs;
    # it does not move the target or prescribe a transit schedule.
    MAX_SIMULATION_TIME = 3.0 * PATH_END_TIME + GOAL_HOLD_TIME
    MAX_STEPS = int(np.ceil(MAX_SIMULATION_TIME / DT))
    settled_steps = 0
    required_settled_steps = int(np.ceil(GOAL_HOLD_TIME / DT))
    trace = []
    progress_interval = max(1, MAX_STEPS // 10)
    for simulation_step in range(MAX_STEPS):
        sample = plant.step(run_controller)
        trace.append(sample)
        # Stop on the first violated constraint; later steps cannot repair this trial
        # and only delay the existing diagnostic raised below.
        if not sample["feasible"]:
            break
        if (simulation_step + 1) % progress_interval == 0:
            print(f"PID simulation progress: {simulation_step + 1}/{MAX_STEPS} steps "
                  f"({100.0 * (simulation_step + 1) / MAX_STEPS:.0f}%)")
        at_route_end = run_controller.path_arc_progress >= PATH_ARC[-1] - 0.02
        at_rest = np.linalg.norm(plant.state[3:5]) < 0.05
        settled_steps = settled_steps + 1 if at_route_end and at_rest else 0
        if settled_steps >= required_settled_steps:
            break
    route_complete = settled_steps >= required_settled_steps
else:
    # MPC retains the original fixed-time demonstration while it is paused.
    SIMULATION_TIME = PATH_END_TIME + GOAL_HOLD_TIME
    trace = plant.run(run_controller, int(np.ceil(SIMULATION_TIME / DT)))
    route_complete = None
N_STEPS = len(trace)
SIMULATION_TIME = N_STEPS * DT
TIME = DT * np.arange(N_STEPS + 1)
state_history = np.vstack((plant.initial_state,
                           np.array([sample["state_after"] for sample in trace])))
if CONTROLLER_MODE.upper() == "PID":
    # Plot the online path-following targets recorded by PID at each state.
    pid_position_targets = np.array([sample["decision"]["position_ref"] for sample in trace])
    pid_velocity_targets = np.array([sample["decision"]["velocity_ref"] for sample in trace])
    pid_acceleration_targets = np.array([sample["decision"]["acceleration_ref"] for sample in trace])
    position_reference_history = np.vstack((pid_position_targets, pid_position_targets[-1]))
    velocity_reference_history = np.vstack((pid_velocity_targets, pid_velocity_targets[-1]))
    acceleration_reference_history = pid_acceleration_targets
else:
    # MPC retains its fixed, time-indexed trajectory reference.
    position_reference_history = np.array([trajectory_reference(t)[0] for t in TIME])
    velocity_reference_history = np.array([trajectory_reference(t)[1] for t in TIME])
    acceleration_reference_history = np.array([trajectory_reference(t)[2] for t in TIME[:-1]])
position_error_history = position_reference_history - state_history[:, :2]
velocity_history = state_history[:, 3:5]
speed_history = np.linalg.norm(velocity_history, axis=1)

acceleration_history = np.array([
    sample["state_derivative"][3:5] for sample in trace
])
# PID uses the locally projected, rate-limited acceleration targets above.
acceleration_magnitude_history = np.linalg.norm(acceleration_history, axis=1)
current_history = np.array([sample["current"] for sample in trace])
commanded_current_history = np.array([sample["decision"]["commanded_current"] for sample in trace])
vq_history = np.array([sample["decision"]["vq_average_V"] for sample in trace])
voltage_saturation_history = np.array([sample["decision"]["voltage_saturation_fraction"] for sample in trace])
traction_history = np.array([sample["required_traction"] for sample in trace])
normal_load_history = np.array([sample["normal_loads_N"] for sample in trace])
traction_limit_history = np.array([sample["wheel_traction_limits_N"] for sample in trace])
slip_onset_history = np.array([sample["slip_onset"] for sample in trace])
torque_residual_history = np.array([sample["wheel_torque_balance_residual"] for sample in trace])
traction_utilization_history = 100.0 * np.abs(traction_history) / np.maximum(traction_limit_history, 1e-12)
drive_slip_history = np.array([sample["drive_slip"] for sample in trace])
drive_force_history = np.array([sample["force_d"] for sample in trace])
axial_slip_history = np.array([sample["slip_a"] for sample in trace])
axial_force_history = np.array([sample["force_a"] for sample in trace])
wheel_speed_history = state_history[:, 6:10]
wheel_angle_history = state_history[:, 10:14]
wheel_contact_history = np.array([sample["wheel_contact"] for sample in trace], dtype=bool)
wheel_speed_rolling_history = np.array([
    -(plant.H.T @ np.r_[world_to_body(state[3:5], state[2]), state[5]]) / WHEEL_RADIUS
    for state in state_history
])
wheel_speed_mismatch_history = wheel_speed_history - wheel_speed_rolling_history

# Recorded issue to revisit: the earlier MPC run hit the obstacle check at k=176 (1.760 s).
# This PID run can also fail the plant safety assertion on collision; PID has no QP status.
infeasible_samples = [sample for sample in trace if not sample["feasible"]]
if infeasible_samples:
    first_bad = infeasible_samples[0]
    violated_constraints = {
        name: float(np.min(margin))
        for name, margin in first_bad["constraint_margins"].items()
        if np.min(margin) < -MPC_CONSTRAINT_TOL
    }
    raise AssertionError(
        f"{len(infeasible_samples)} infeasible plant samples; first at "
        f"k={first_bad['step']} t={first_bad['time']:.3f}s, "
        f"current={np.max(np.abs(first_bad['current'])):.4f}/{IQ_LIMIT:.4f} A, "
        f"margin={first_bad['minimum_constraint_margin']:.3e}, "
        f"yaw_rate={abs(first_bad['state_after'][5]):.4f}/{MAX_YAW_RATE:.4f} rad/s, "
        f"violated_constraints={violated_constraints}, "
        f"obstacle_collision={first_bad['collision']}, "
        f"obstacle_clearance_margin={first_bad['minimum_obstacle_clearance_margin']:.4f} m"
    )
assert max(np.max(np.abs(sample["force_balance_residual"])) for sample in trace) < 1e-8
assert max(np.max(np.abs(sample["wheel_torque_balance_residual"])) for sample in trace) < 1e-10
assert np.max(np.abs(current_history)) <= IQ_LIMIT + MPC_CONSTRAINT_TOL
if not ALLOW_DRIVE_TRACTION_OVERLOAD:
    assert np.all(np.abs(traction_history) <= traction_limit_history + MPC_CONSTRAINT_TOL)
if CONTROLLER_MODE.upper() == "PID":
    achieved_progress = run_controller.path_arc_progress
    print(f"\nPID route progress: {achieved_progress:.2f}/{PATH_ARC[-1]:.2f} m; "
          f"settled at endpoint: {route_complete}; run time: {SIMULATION_TIME:.2f} s")
else:
    print(f"\nMPC reference transit time: {PATH_END_TIME:.2f} s; "
          f"simulation including goal hold: {SIMULATION_TIME:.2f} s")
print(f"{CONTROLLER_MODE.upper()} run: {N_STEPS} explicit plant steps")
print(f"Position RMS error: {np.sqrt(np.mean(position_error_history**2)):.5f} m per axis")
print(f"Maximum position error: {np.max(np.linalg.norm(position_error_history, axis=1)):.5f} m")
print(f"Peak actual wheel current: {np.max(np.abs(current_history)):.3f} A / {IQ_LIMIT:.3f} A")
print(f"Peak commanded q-current: {np.max(np.abs(commanded_current_history)):.3f} A")
print(f"Peak average q-voltage: {np.max(np.abs(vq_history)):.3f} V / {FOC_Q_AXIS_VOLTAGE_LIMIT:.3f} V limit")
print(f"Mean q-voltage saturation fraction: {np.mean(voltage_saturation_history):.3%}")
print(f"Peak required drive traction: {np.max(np.abs(traction_history)):.3f} N; "
      f"nominal equal-load limit: {F_ROLL_LIMIT:.3f} N/wheel")
print(f"Wheel-load range: {np.min(normal_load_history):.3f} to "
      f"{np.max(normal_load_history):.3f} N")
print(f"Peak traction utilization: {np.max(traction_utilization_history):.1f}%")
print(f"Drive-slip onset samples (threshold { SLIP_THRESHOLD:.4f} m/s): "
      f"{int(np.sum(np.any(slip_onset_history, axis=1)))}")
print(f"Peak drive-direction slip: {np.max(np.abs(drive_slip_history)):.4f} m/s")
print(f"Peak passive-roller axial slip: {np.max(np.abs(axial_slip_history)):.4f} m/s")
print(f"Peak passive-roller axial friction: {np.max(np.abs(axial_force_history)):.3f} N/wheel")
print(f"Peak speed: {np.max(speed_history):.3f} m/s")
print(f"Peak linear acceleration: {np.max(acceleration_magnitude_history):.3f} m/s^2")
print(f"Peak wheel torque-balance residual: {np.max(np.abs(torque_residual_history)):.2e} N m")

# Dana-style experiment summary dashboard (adapted to the project FOC actuator).
summary_fig, summary_axes = plt.subplots(2, 2, figsize=(14, 10))
summary_fig.suptitle("FOC current-controlled wheel-slip run", fontsize=14, fontweight="bold")

summary_axes[0, 0].plot(position_reference_history[:, 0], position_reference_history[:, 1],
                        color="#333333", linestyle="--", linewidth=1.8, label="Reference")
summary_axes[0, 0].plot(state_history[:, 0], state_history[:, 1],
                        color="#1769aa", linewidth=1.8, label="Actual")
summary_axes[0, 0].set(title="Global trajectory", xlabel="x [m]", ylabel="y [m]")
summary_axes[0, 0].set_aspect("equal", adjustable="box")
summary_axes[0, 0].grid(True, alpha=0.3)
summary_axes[0, 0].legend()

summary_axes[0, 1].plot(TIME, position_error_history[:, 0], label="x error")
summary_axes[0, 1].plot(TIME, position_error_history[:, 1], label="y error")
summary_axes[0, 1].set(title="Position tracking error", xlabel="Time [s]", ylabel="Error [m]")
summary_axes[0, 1].grid(True, alpha=0.3)
summary_axes[0, 1].legend()

for wheel in range(N_WHEELS):
    actual_line, = summary_axes[1, 0].plot(
        TIME[:-1], current_history[:, wheel], linewidth=1.5, label=f"Wheel {wheel + 1} actual")
    summary_axes[1, 0].plot(
        TIME[:-1], commanded_current_history[:, wheel], color=actual_line.get_color(),
        linestyle="--", alpha=0.7, linewidth=1.1, label=f"Wheel {wheel + 1} command")
summary_axes[1, 0].set(title="FOC q-axis current: actual and commanded",
                       xlabel="Time [s]", ylabel="Current [A]")
summary_axes[1, 0].axhline(IQ_LIMIT, color="black", linestyle=":", linewidth=1)
summary_axes[1, 0].axhline(-IQ_LIMIT, color="black", linestyle=":", linewidth=1)
summary_axes[1, 0].grid(True, alpha=0.3)
summary_axes[1, 0].legend(ncol=2, fontsize=8)

for wheel in range(N_WHEELS):
    summary_axes[1, 1].plot(TIME[:-1], vq_history[:, wheel], linewidth=1.3,
                            label=f"Wheel {wheel + 1}")
summary_axes[1, 1].axhline(FOC_Q_AXIS_VOLTAGE_LIMIT, color="black", linestyle=":",
                           linewidth=1, label="Voltage limit")
summary_axes[1, 1].axhline(-FOC_Q_AXIS_VOLTAGE_LIMIT, color="black", linestyle=":", linewidth=1)
summary_axes[1, 1].set(title="FOC q-axis voltage", xlabel="Time [s]", ylabel="Voltage [V]")
summary_axes[1, 1].grid(True, alpha=0.3)
summary_axes[1, 1].legend(ncol=2, fontsize=8)

summary_fig.tight_layout()
display(summary_fig)
plt.close(summary_fig)

# Explicit IPython display keeps all plots inline in the notebook output.
from IPython.display import HTML, display

# The animated field view is the primary simulation visualization.
# Animated top-down XY view. This silhouette is for display; collision checks
# still use the separate ROBOT_CLEARANCE value in the simulator/MPC.
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.transforms import Affine2D
from IPython.display import HTML, display

# Follow the COCO_T1 solution's interactive XY animation: fixed field axes,
# an animated robot trace, a visible reference path, and play/scrub controls.
# Keep this plot inline in the notebook, as in the COCO example.
import plotly.graph_objects as go

# The robot uses the project wheel/contact radius and the SSL-style clipped front.
ROBOT_BODY_RADIUS = WHEEL_ROBOT_DIST
SSL_FRONT_CUT_FRACTION = 0.055 / 0.085
ROBOT_FRONT_CUT_X = SSL_FRONT_CUT_FRACTION * ROBOT_BODY_RADIUS
cut_angle = np.arccos(ROBOT_FRONT_CUT_X / ROBOT_BODY_RADIUS)
arc_angles = np.linspace(cut_angle, 2.0 * np.pi - cut_angle, 100)
robot_outline_local = np.vstack((
    np.column_stack((ROBOT_BODY_RADIUS * np.cos(arc_angles),
                     ROBOT_BODY_RADIUS * np.sin(arc_angles))),
    [ROBOT_FRONT_CUT_X, ROBOT_BODY_RADIUS * np.sin(cut_angle)],
))
front_edge_local = np.array([
    [ROBOT_FRONT_CUT_X, -ROBOT_BODY_RADIUS * np.sin(cut_angle)],
    [ROBOT_FRONT_CUT_X,  ROBOT_BODY_RADIUS * np.sin(cut_angle)],
])

# Downsample long runs for a responsive animation while retaining both endpoints.
frame_indices = np.unique(np.linspace(
    0, len(TIME) - 1, min(240, len(TIME)), dtype=int
))

def robot_world_geometry(sample_index):
    """Rotate and translate the robot outline using the simulated pose."""
    x, y, phi = state_history[sample_index, :3]
    rotation = np.array([[np.cos(phi), -np.sin(phi)],
                         [np.sin(phi),  np.cos(phi)]])
    outline = robot_outline_local @ rotation.T + np.array([x, y])
    front = front_edge_local @ rotation.T + np.array([x, y])
    nose = np.array([x, y]) + ROBOT_BODY_RADIUS * np.array([np.cos(phi), np.sin(phi)])
    return x, y, phi, outline, front, nose

first_index = int(frame_indices[0])
x0, y0, phi0, outline0, front0, nose0 = robot_world_geometry(first_index)
static_traces = []
for obstacle_index, obstacle_vertices in enumerate(STATIC_OBSTACLES):
    vertices = np.asarray(obstacle_vertices, dtype=float)
    closed = np.vstack((vertices, vertices[0]))
    static_traces.append(go.Scatter(
        x=closed[:, 0], y=closed[:, 1], mode="lines", fill="toself",
        fillcolor="rgba(120,120,120,0.48)",
        line=dict(color="#444444", width=1.5),
        name="static obstacle" if obstacle_index == 0 else None,
        showlegend=(obstacle_index == 0), hoverinfo="skip",
    ))

obstacle_trace_count = len(static_traces)
path_trace = go.Scatter(
    x=_route_points[:, 0], y=_route_points[:, 1],
    mode="lines", line=dict(color="gray", width=1.5, dash="dot"),
    name="route centerline",
)
trail_trace = go.Scatter(
    x=state_history[:first_index + 1, 0], y=state_history[:first_index + 1, 1],
    mode="lines", line=dict(color="#0072B2", width=2.5), name="robot path",
)
target_trace = go.Scatter(
    x=[position_reference_history[first_index, 0]],
    y=[position_reference_history[first_index, 1]],
    mode="markers", marker=dict(symbol="x", size=10, color="#D55E00"),
    name="current target",
)
robot_trace = go.Scatter(
    x=np.r_[outline0[:, 0], outline0[0, 0]],
    y=np.r_[outline0[:, 1], outline0[0, 1]],
    mode="lines", fill="toself", fillcolor="rgba(0,114,178,0.82)",
    line=dict(color="black", width=2), name="robot (to scale)",
)
front_trace = go.Scatter(
    x=front0[:, 0], y=front0[:, 1], mode="lines",
    line=dict(color="crimson", width=4), name="robot front / heading",
)
heading_trace = go.Scatter(
    x=[x0, nose0[0]], y=[y0, nose0[1]], mode="lines",
    line=dict(color="crimson", width=2), showlegend=False, hoverinfo="skip",
)
start_trace = go.Scatter(
    x=[ROUTE_WAYPOINTS[0, 0]], y=[ROUTE_WAYPOINTS[0, 1]],
    mode="markers", marker=dict(symbol="x", size=10, color="green"),
    name="start / stop",
)
traces = static_traces + [path_trace, trail_trace, target_trace,
                          robot_trace, front_trace, heading_trace, start_trace]
path_i = obstacle_trace_count
trail_i, target_i = path_i + 1, path_i + 2
robot_i, front_i, heading_i, start_i = path_i + 3, path_i + 4, path_i + 5, path_i + 6

frames = []
slider_steps = []
for frame_number, sample_index in enumerate(frame_indices):
    sample_index = int(sample_index)
    x, y, phi, outline, front, nose = robot_world_geometry(sample_index)
    frame_name = str(frame_number)
    speed = np.linalg.norm(state_history[sample_index, 3:5])
    frame_data = [
        go.Scatter(x=state_history[:sample_index + 1, 0],
                   y=state_history[:sample_index + 1, 1]),
        go.Scatter(x=[position_reference_history[sample_index, 0]],
                   y=[position_reference_history[sample_index, 1]]),
        go.Scatter(x=np.r_[outline[:, 0], outline[0, 0]],
                   y=np.r_[outline[:, 1], outline[0, 1]]),
        go.Scatter(x=front[:, 0], y=front[:, 1]),
        go.Scatter(x=[x, nose[0]], y=[y, nose[1]]),
    ]
    frame_layout = go.Layout(title=(
        f"4.5 m x 6 m field | t={TIME[sample_index]:.2f} s | "
        f"heading={np.degrees(phi):.1f} deg | speed={speed:.2f} m/s"
    ))
    frames.append(go.Frame(data=frame_data,
                           traces=[trail_i, target_i, robot_i, front_i, heading_i],
                           layout=frame_layout, name=frame_name))
    slider_steps.append(dict(
        label=f"{TIME[sample_index]:.2f}", method="animate",
        args=[[frame_name], dict(mode="immediate",
                                 frame=dict(duration=0, redraw=True),
                                 transition=dict(duration=0))],
    ))

anim_fig = go.Figure(data=traces, frames=frames)
anim_fig.update_layout(
    title=(f"4.5 m x 6 m field | t={TIME[first_index]:.2f} s | "
           f"heading={np.degrees(phi0):.1f} deg | "
           f"speed={np.linalg.norm(state_history[first_index, 3:5]):.2f} m/s"),
    width=900, height=760, template="plotly_white",
    xaxis=dict(title="x [m]", range=[-FIELD_LENGTH / 2, FIELD_LENGTH / 2],
               showgrid=True, gridcolor="#d8e0dd", zeroline=False),
    yaxis=dict(title="y [m]", range=[-FIELD_WIDTH / 2, FIELD_WIDTH / 2],
               scaleanchor="x", scaleratio=1, showgrid=True,
               gridcolor="#d8e0dd", zeroline=False),
    plot_bgcolor="#f1f5f3", hovermode="closest",
    legend=dict(x=1.02, y=1.0),
    updatemenus=[dict(type="buttons", x=0.0, y=1.12, xanchor="left",
                      direction="right", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=max(20, int(1000 * SIMULATION_TIME / max(1, len(frame_indices)))), redraw=True),
                              fromcurrent=True, transition=dict(duration=0))]),
        dict(label="⏸ Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(active=0, currentvalue=dict(prefix="time [s] = "),
                  steps=slider_steps)],
    margin=dict(l=70, r=190, t=105, b=80),
)
# COCO's XY view uses equal x/y scale. The field outline confirms the exact board size.
anim_fig.add_shape(type="rect", x0=-FIELD_LENGTH / 2, y0=-FIELD_WIDTH / 2,
                   x1=FIELD_LENGTH / 2, y1=FIELD_WIDTH / 2,
                   line=dict(color="#333333", width=2), fillcolor="rgba(0,0,0,0)")
anim_fig.add_shape(type="line", x0=0.0, y0=-FIELD_WIDTH / 2,
                   x1=0.0, y1=FIELD_WIDTH / 2,
                   line=dict(color="white", width=2))
display(anim_fig)  # Notebook display keeps the interactive plot inline.

# Scalar plots below are secondary diagnostics and remain inline as well.
figure, axes = plt.subplots(6, 2, figsize=(14, 24))
axes[0, 0].plot(position_reference_history[:, 0], position_reference_history[:, 1], "k--", label="reference")
axes[0, 0].plot(state_history[:, 0], state_history[:, 1], label=CONTROLLER_MODE.upper())
axes[0, 0].set_aspect("equal", adjustable="box")
axes[0, 0].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[0, 0].set_title("Figure-eight position trajectory")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(TIME, position_error_history[:, 0], label="x error")
axes[0, 1].plot(TIME, position_error_history[:, 1], label="y error")
axes[0, 1].set_ylabel("Position error [m]")
axes[0, 1].set_title("Tracking error")
axes[0, 1].legend()
axes[0, 1].grid(True)

for wheel in range(N_WHEELS):
    axes[1, 0].plot(TIME[:-1], current_history[:, wheel], label=f"wheel {wheel + 1}")
axes[1, 0].set_ylim(-IQ_LIMIT, IQ_LIMIT)
axes[1, 0].axhline(IQ_LIMIT, color="black", linestyle=":")
axes[1, 0].axhline(-IQ_LIMIT, color="black", linestyle=":")
axes[1, 0].set_ylabel("Current [A]")
axes[1, 0].set_title(f"Wheel current (limit +/-{IQ_LIMIT:g} A)")
axes[1, 0].legend(ncol=2)
axes[1, 0].grid(True)

for wheel in range(N_WHEELS):
    line, = axes[1, 1].plot(TIME[:-1], drive_force_history[:, wheel], label=f"wheel {wheel + 1}")
    axes[1, 1].plot(TIME[:-1], traction_limit_history[:, wheel],
                    color=line.get_color(), linestyle=":", alpha=0.7)
    axes[1, 1].plot(TIME[:-1], -traction_limit_history[:, wheel],
                    color=line.get_color(), linestyle=":", alpha=0.7)
traction_plot_limit = max(F_ROLL_LIMIT, float(np.max(traction_limit_history)))
axes[1, 1].set_ylim(-1.1 * traction_plot_limit, 1.1 * traction_plot_limit)
axes[1, 1].set_ylabel("Drive contact force [N]")
axes[1, 1].set_title("Project d-axis contact force and static capacity (dotted)")
axes[1, 1].legend(ncol=2)
axes[1, 1].grid(True)

for wheel in range(N_WHEELS):
    axes[2, 0].plot(TIME[:-1], traction_utilization_history[:, wheel], label=f"wheel {wheel + 1}")
axes[2, 0].axhline(100.0, color="black", linestyle=":", label="static capacity")
axes[2, 0].set_ylim(0.0, 105.0)
axes[2, 0].set_xlabel("Time [s]")
axes[2, 0].set_ylabel("Contact-force capacity used [%]")
axes[2, 0].set_title("Drive contact-force utilization")
axes[2, 0].legend(ncol=2)
axes[2, 0].grid(True)

axial_force_ax = axes[2, 1].twinx()
for wheel in range(N_WHEELS):
    axes[2, 1].plot(TIME[:-1], axial_slip_history[:, wheel], label=f"wheel {wheel + 1} slip")
    axial_force_ax.plot(TIME[:-1], axial_force_history[:, wheel], linestyle="--",
                        label=f"wheel {wheel + 1} force")
axes[2, 1].axhline(0.0, color="0.35", linewidth=0.8)
axes[2, 1].set_xlabel("Time [s]")
axes[2, 1].set_ylabel("Axial roller slip [m/s]")
axial_force_ax.set_ylabel("Axial friction force [N]")
axes[2, 1].set_title("Project a-axis passive-roller slip and force")
slip_handles, slip_labels = axes[2, 1].get_legend_handles_labels()
force_handles, force_labels = axial_force_ax.get_legend_handles_labels()
axes[2, 1].legend(slip_handles + force_handles, slip_labels + force_labels,
                  ncol=2, fontsize=7)
axes[2, 1].grid(True)

drive_force_ax = axes[3, 0].twinx()
for wheel in range(N_WHEELS):
    axes[3, 0].plot(TIME[:-1], drive_slip_history[:, wheel], label=f"wheel {wheel + 1} slip")
    drive_force_ax.plot(TIME[:-1], drive_force_history[:, wheel], linestyle="--",
                        label=f"wheel {wheel + 1} force")
axes[3, 0].axhline(0.0, color="0.35", linewidth=0.8)
axes[3, 0].axhline(SLIP_THRESHOLD, color="black", linestyle=":", linewidth=0.8)
axes[3, 0].axhline(-SLIP_THRESHOLD, color="black", linestyle=":", linewidth=0.8)
axes[3, 0].set_xlabel("Time [s]")
axes[3, 0].set_ylabel("Drive slip [m/s]")
drive_force_ax.set_ylabel("Drive contact force [N]")
axes[3, 0].set_title("Project d-axis wheel slip and force")
slip_handles, slip_labels = axes[3, 0].get_legend_handles_labels()
force_handles, force_labels = drive_force_ax.get_legend_handles_labels()
axes[3, 0].legend(slip_handles + force_handles, slip_labels + force_labels,
                  ncol=2, fontsize=7)
axes[3, 0].grid(True)

for wheel in range(N_WHEELS):
    axes[3, 1].plot(TIME, wheel_speed_history[:, wheel], label=f"wheel {wheel + 1} actual")
    axes[3, 1].plot(TIME, wheel_speed_rolling_history[:, wheel], linestyle="--",
                    alpha=0.7, label=f"wheel {wheel + 1} rolling")
axes[3, 1].set_xlabel("Time [s]")
axes[3, 1].set_ylabel("Wheel angular velocity [rad/s]")
axes[3, 1].set_title("Independent wheel speed and no-slip reference")
axes[3, 1].legend(ncol=2, fontsize=7)
axes[3, 1].grid(True)

# Linear velocity
axes[4, 0].plot(TIME, velocity_history[:, 0], label="vx actual")
axes[4, 0].plot(TIME, velocity_reference_history[:, 0], "--", label="vx reference")
axes[4, 0].plot(TIME, velocity_history[:, 1], label="vy actual")
axes[4, 0].plot(TIME, velocity_reference_history[:, 1], "--", label="vy reference")
axes[4, 0].plot(TIME, speed_history, label="speed magnitude")
axes[4, 0].set_xlabel("Time [s]")
axes[4, 0].set_ylabel("Velocity [m/s]")
axes[4, 0].set_title("Linear velocity")
axes[4, 0].legend(ncol=2)
axes[4, 0].grid(True)

# Linear acceleration
axes[4, 1].plot(TIME[:-1], acceleration_history[:, 0], label="ax actual")
axes[4, 1].plot(TIME[:-1], acceleration_reference_history[:, 0], "--", label="ax reference")
axes[4, 1].plot(TIME[:-1], acceleration_history[:, 1], label="ay actual")
axes[4, 1].plot(TIME[:-1], acceleration_reference_history[:, 1], "--", label="ay reference")
axes[4, 1].plot(TIME[:-1], acceleration_magnitude_history, label="|a| actual")
axes[4, 1].axhline(MAX_ACCELERATION, color="black", linestyle=":", label="acceleration limit")
axes[4, 1].set_xlabel("Time [s]")
axes[4, 1].set_ylabel("Acceleration [m/s²]")
axes[4, 1].set_title("Linear acceleration")
axes[4, 1].legend(ncol=2)
axes[4, 1].grid(True)

for wheel in range(N_WHEELS):
    axes[5, 0].plot(TIME, wheel_angle_history[:, wheel], label=f"wheel {wheel + 1}")
axes[5, 0].set_xlabel("Time [s]")
axes[5, 0].set_ylabel("Wheel angle [rad]")
axes[5, 0].set_title("Integrated wheel angles")
axes[5, 0].legend(ncol=2)
axes[5, 0].grid(True)

for wheel in range(N_WHEELS):
    axes[5, 1].step(TIME[:-1], wheel_contact_history[:, wheel].astype(int),
                    where="post", label=f"wheel {wheel + 1}")
axes[5, 1].set_xlabel("Time [s]")
axes[5, 1].set_ylabel("Contact state (1=roller, 0=gap)")
axes[5, 1].set_title("Binary roller/gap contact")
axes[5, 1].set_yticks([0, 1])
axes[5, 1].legend(ncol=2)
axes[5, 1].grid(True)

figure.tight_layout()
display(figure)  # Render in the notebook output instead of opening a GUI window.
plt.close(figure)




In [ ]:
# Run a short MPC check against the same slip-capable plant when MPC is selected.
if CONTROLLER_MODE.upper() == "MPC":
    mpc_plant = WheelSlipRobotSimulator()
    mpc_controller = ConvexMPCController(horizon=2)
    mpc_snapshot = mpc_plant.step(mpc_controller)

    assert mpc_snapshot["decision"]["qp_status"] in ("optimal", "optimal_inaccurate")
    assert mpc_snapshot["decision"]["planned_states"].shape == (14, 3)
    assert mpc_snapshot["decision"]["planned_currents"].shape == (N_WHEELS, 2)
    assert mpc_snapshot["feasible"]

    print("QP status:", mpc_snapshot["decision"]["qp_status"])
    print("Objective:", round(mpc_snapshot["decision"]["qp_objective"], 6))
    print("First current decision [A]:", np.round(mpc_snapshot["current"], 5))
    print("Predicted state/current shapes:",
          mpc_snapshot["decision"]["planned_states"].shape,
          mpc_snapshot["decision"]["planned_currents"].shape)
    print("Minimum plant constraint margin:", round(mpc_snapshot["minimum_constraint_margin"], 6))
else:
    print("Standalone MPC demo skipped because CONTROLLER_MODE is PID.")